# SOFR, FX & Cross-Currency Curves · Episode 1 — The USD money market

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **Data.** `data/nyfed_effr_sofr.csv` holds the effective federal funds rate (EFFR) with the FOMC target range and volume, and the Secured Overnight Financing Rate (SOFR) with volume, from the New York Fed's markets data API (https://markets.newyorkfed.org).
>
> The SOFR and EFFR data are subject to the Terms of Use posted at newyorkfed.org. The New York Fed is not responsible for publication of the SOFR and EFFR data by Rates with Bip, does not sanction or endorse any particular republication, and has no liability for your use. The SOFR data are calculated using data provided under a license granted to the New York Fed by DTCC Solutions LLC ("Solutions"), an affiliate of The Depository Trust & Clearing Corporation; Solutions, its affiliates, and third parties from which they obtained data have no liability for the content of this material.
>
> Rates with Bip is not affiliated with the New York Fed. The New York Fed does not sanction, endorse, or recommend any products or services offered by Rates with Bip.
>
> Educational material only, not investment advice.

1. Setup · 2. The Fed's settings · 3. Two overnight rates · 4. The target range since 2018 · 5. EFFR inside the range · 6. SOFR spikes · 7. A 360-day year · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `data/nyfed_effr_sofr.csv` (run me first) { display-mode: "form" }
# Source: Federal Reserve Bank of New York (SOFR, EFFR). The SOFR and EFFR data are subject to the Terms of
# Use posted at newyorkfed.org. The New York Fed is not responsible for publication of the SOFR and EFFR data
# by Rates with Bip, does not sanction or endorse any particular republication, and has no liability for
# your use. SOFR data are calculated using data provided under a license granted to the New York Fed by
# DTCC Solutions LLC; Solutions, its affiliates, and third parties from which they obtained data have no
# liability for the content of this material. Rates with Bip is not affiliated with the New York Fed.
from pathlib import Path
Path('data/nyfed_effr_sofr.csv').parent.mkdir(parents=True, exist_ok=True)
Path('data/nyfed_effr_sofr.csv').write_text("""date,effr,target_low,target_high,effr_volume_bn,sofr,sofr_volume_bn
2018-04-02,1.68,1.5,1.75,89,1.8,849.0
2018-04-03,1.69,1.5,1.75,90,1.83,825.0
2018-04-04,1.69,1.5,1.75,83,1.74,859.0
2018-04-05,1.69,1.5,1.75,88,1.75,829.0
2018-04-06,1.69,1.5,1.75,78,1.75,845.0
2018-04-09,1.69,1.5,1.75,78,1.75,833.0
2018-04-10,1.69,1.5,1.75,80,1.75,811.0
2018-04-11,1.69,1.5,1.75,74,1.76,811.0
2018-04-12,1.69,1.5,1.75,83,1.73,871.0
2018-04-13,1.69,1.5,1.75,74,1.72,776.0
2018-04-16,1.69,1.5,1.75,82,1.77,770.0
2018-04-17,1.69,1.5,1.75,66,1.76,729.0
2018-04-18,1.69,1.5,1.75,63,1.75,738.0
2018-04-19,1.69,1.5,1.75,75,1.73,745.0
2018-04-20,1.7,1.5,1.75,72,1.72,726.0
2018-04-23,1.7,1.5,1.75,67,1.7,755.0
2018-04-24,1.7,1.5,1.75,63,1.71,747.0
2018-04-25,1.7,1.5,1.75,71,1.71,728.0
2018-04-26,1.7,1.5,1.75,77,1.72,725.0
2018-04-27,1.7,1.5,1.75,77,1.72,732.0
2018-04-30,1.69,1.5,1.75,76,1.77,775.0
2018-05-01,1.7,1.5,1.75,81,1.76,761.0
2018-05-02,1.7,1.5,1.75,76,1.75,753.0
2018-05-03,1.7,1.5,1.75,82,1.74,759.0
2018-05-04,1.7,1.5,1.75,83,1.72,752.0
2018-05-07,1.7,1.5,1.75,85,1.71,760.0
2018-05-08,1.7,1.5,1.75,83,1.72,746.0
2018-05-09,1.7,1.5,1.75,89,1.72,783.0
2018-05-10,1.7,1.5,1.75,89,1.73,737.0
2018-05-11,1.7,1.5,1.75,88,1.73,746.0
2018-05-14,1.7,1.5,1.75,79,1.75,741.0
2018-05-15,1.7,1.5,1.75,83,1.79,770.0
2018-05-16,1.7,1.5,1.75,85,1.75,754.0
2018-05-17,1.7,1.5,1.75,80,1.74,754.0
2018-05-18,1.7,1.5,1.75,78,1.73,750.0
2018-05-21,1.7,1.5,1.75,80,1.69,772.0
2018-05-22,1.7,1.5,1.75,89,1.65,748.0
2018-05-23,1.7,1.5,1.75,92,1.67,729.0
2018-05-24,1.7,1.5,1.75,90,1.74,772.0
2018-05-25,1.7,1.5,1.75,92,1.73,749.0
2018-05-29,1.7,1.5,1.75,89,1.72,796.0
2018-05-30,1.7,1.5,1.75,83,1.72,801.0
2018-05-31,1.7,1.5,1.75,84,1.81,872.0
2018-06-01,1.7,1.5,1.75,94,1.81,807.0
2018-06-04,1.7,1.5,1.75,92,1.8,825.0
2018-06-05,1.7,1.5,1.75,90,1.75,797.0
2018-06-06,1.7,1.5,1.75,94,1.73,805.0
2018-06-07,1.7,1.5,1.75,87,1.71,813.0
2018-06-08,1.7,1.5,1.75,90,1.69,806.0
2018-06-11,1.7,1.5,1.75,75,1.69,791.0
2018-06-12,1.7,1.5,1.75,81,1.67,774.0
2018-06-13,1.7,1.5,1.75,80,1.71,817.0
2018-06-14,1.9,1.75,2.0,80,1.9,818.0
2018-06-15,1.9,1.75,2.0,69,1.91,802.0
2018-06-18,1.9,1.75,2.0,61,1.9,799.0
2018-06-19,1.91,1.75,2.0,59,1.88,799.0
2018-06-20,1.92,1.75,2.0,63,1.87,769.0
2018-06-21,1.92,1.75,2.0,81,1.87,753.0
2018-06-22,1.92,1.75,2.0,78,1.92,742.0
2018-06-25,1.92,1.75,2.0,76,1.91,755.0
2018-06-26,1.92,1.75,2.0,82,1.9,757.0
2018-06-27,1.91,1.75,2.0,83,1.9,760.0
2018-06-28,1.91,1.75,2.0,79,1.93,768.0
2018-06-29,1.91,1.75,2.0,79,2.12,702.0
2018-07-02,1.91,1.75,2.0,83,2.04,775.0
2018-07-03,1.91,1.75,2.0,88,2.0,768.0
2018-07-05,1.91,1.75,2.0,78,1.97,807.0
2018-07-06,1.91,1.75,2.0,80,1.93,760.0
2018-07-09,1.91,1.75,2.0,77,1.89,784.0
2018-07-10,1.91,1.75,2.0,80,1.89,762.0
2018-07-11,1.91,1.75,2.0,82,1.89,766.0
2018-07-12,1.91,1.75,2.0,84,1.9,747.0
2018-07-13,1.91,1.75,2.0,79,1.9,735.0
2018-07-16,1.91,1.75,2.0,80,1.95,767.0
2018-07-17,1.91,1.75,2.0,78,1.92,766.0
2018-07-18,1.91,1.75,2.0,78,1.9,783.0
2018-07-19,1.91,1.75,2.0,72,1.9,769.0
2018-07-20,1.91,1.75,2.0,77,1.88,773.0
2018-07-23,1.91,1.75,2.0,71,1.87,764.0
2018-07-24,1.91,1.75,2.0,74,1.9,775.0
2018-07-25,1.91,1.75,2.0,77,1.87,753.0
2018-07-26,1.91,1.75,2.0,74,1.9,784.0
2018-07-27,1.91,1.75,2.0,79,1.88,774.0
2018-07-30,1.91,1.75,2.0,74,1.88,768.0
2018-07-31,1.91,1.75,2.0,67,1.93,799.0
2018-08-01,1.91,1.75,2.0,77,1.88,786.0
2018-08-02,1.91,1.75,2.0,80,1.91,781.0
2018-08-03,1.91,1.75,2.0,83,1.86,763.0
2018-08-06,1.91,1.75,2.0,76,1.86,763.0
2018-08-07,1.91,1.75,2.0,68,1.87,732.0
2018-08-08,1.91,1.75,2.0,70,1.88,738.0
2018-08-09,1.91,1.75,2.0,73,1.91,761.0
2018-08-10,1.91,1.75,2.0,82,1.9,735.0
2018-08-13,1.91,1.75,2.0,73,1.91,733.0
2018-08-14,1.91,1.75,2.0,68,1.93,729.0
2018-08-15,1.91,1.75,2.0,66,1.98,807.0
2018-08-16,1.92,1.75,2.0,64,1.99,782.0
2018-08-17,1.92,1.75,2.0,55,1.9,799.0
2018-08-20,1.92,1.75,2.0,56,1.9,778.0
2018-08-21,1.92,1.75,2.0,59,1.9,772.0
2018-08-22,1.92,1.75,2.0,64,1.9,787.0
2018-08-23,1.92,1.75,2.0,68,1.94,773.0
2018-08-24,1.92,1.75,2.0,71,1.95,773.0
2018-08-27,1.92,1.75,2.0,69,1.95,814.0
2018-08-28,1.92,1.75,2.0,64,1.95,805.0
2018-08-29,1.92,1.75,2.0,65,1.93,797.0
2018-08-30,1.92,1.75,2.0,69,1.93,795.0
2018-08-31,1.91,1.75,2.0,78,1.97,804.0
2018-09-04,1.92,1.75,2.0,73,1.95,833.0
2018-09-05,1.92,1.75,2.0,78,1.95,798.0
2018-09-06,1.92,1.75,2.0,76,1.94,799.0
2018-09-07,1.92,1.75,2.0,78,1.94,802.0
2018-09-10,1.92,1.75,2.0,74,1.94,816.0
2018-09-11,1.92,1.75,2.0,75,1.94,802.0
2018-09-12,1.92,1.75,2.0,75,1.94,810.0
2018-09-13,1.92,1.75,2.0,76,1.94,810.0
2018-09-14,1.92,1.75,2.0,68,1.95,798.0
2018-09-17,1.92,1.75,2.0,62,2.0,789.0
2018-09-18,1.92,1.75,2.0,54,1.94,819.0
2018-09-19,1.92,1.75,2.0,60,1.92,794.0
2018-09-20,1.92,1.75,2.0,69,1.92,808.0
2018-09-21,1.92,1.75,2.0,69,1.92,781.0
2018-09-24,1.93,1.75,2.0,68,1.95,799.0
2018-09-25,1.93,1.75,2.0,67,1.93,814.0
2018-09-26,1.93,1.75,2.0,71,1.92,828.0
2018-09-27,2.18,2.0,2.25,76,2.16,825.0
2018-09-28,2.18,2.0,2.25,70,2.25,807.0
2018-10-01,2.18,2.0,2.25,55,2.22,917.0
2018-10-02,2.18,2.0,2.25,62,2.2,861.0
2018-10-03,2.18,2.0,2.25,67,2.2,855.0
2018-10-04,2.18,2.0,2.25,73,2.18,816.0
2018-10-05,2.18,2.0,2.25,64,2.16,873.0
2018-10-09,2.18,2.0,2.25,57,2.15,855.0
2018-10-10,2.18,2.0,2.25,64,2.15,839.0
2018-10-11,2.18,2.0,2.25,66,2.17,838.0
2018-10-12,2.18,2.0,2.25,69,2.18,852.0
2018-10-15,2.18,2.0,2.25,58,2.21,837.0
2018-10-16,2.18,2.0,2.25,59,2.18,816.0
2018-10-17,2.19,2.0,2.25,54,2.18,771.0
2018-10-18,2.19,2.0,2.25,54,2.19,810.0
2018-10-19,2.19,2.0,2.25,58,2.19,785.0
2018-10-22,2.19,2.0,2.25,54,2.18,791.0
2018-10-23,2.2,2.0,2.25,58,2.17,779.0
2018-10-24,2.2,2.0,2.25,60,2.18,828.0
2018-10-25,2.2,2.0,2.25,67,2.19,822.0
2018-10-26,2.2,2.0,2.25,64,2.19,779.0
2018-10-29,2.2,2.0,2.25,59,2.18,806.0
2018-10-30,2.2,2.0,2.25,60,2.18,788.0
2018-10-31,2.2,2.0,2.25,58,2.22,849.0
2018-11-01,2.2,2.0,2.25,68,2.22,835.0
2018-11-02,2.19,2.0,2.25,74,2.25,853.0
2018-11-05,2.2,2.0,2.25,70,2.24,842.0
2018-11-06,2.2,2.0,2.25,71,2.22,802.0
2018-11-07,2.2,2.0,2.25,67,2.18,823.0
2018-11-08,2.2,2.0,2.25,64,2.21,819.0
2018-11-09,2.19,2.0,2.25,66,2.2,836.0
2018-11-13,2.2,2.0,2.25,60,2.2,817.0
2018-11-14,2.2,2.0,2.25,62,2.2,803.0
2018-11-15,2.2,2.0,2.25,60,2.28,846.0
2018-11-16,2.2,2.0,2.25,69,2.26,828.0
2018-11-19,2.2,2.0,2.25,67,2.24,863.0
2018-11-20,2.2,2.0,2.25,61,2.2,845.0
2018-11-21,2.2,2.0,2.25,59,2.18,819.0
2018-11-23,2.2,2.0,2.25,65,2.23,820.0
2018-11-26,2.2,2.0,2.25,62,2.2,862.0
2018-11-27,2.2,2.0,2.25,57,2.19,843.0
2018-11-28,2.2,2.0,2.25,65,2.19,850.0
2018-11-29,2.2,2.0,2.25,69,2.24,850.0
2018-11-30,2.2,2.0,2.25,75,2.28,919.0
2018-12-03,2.19,2.0,2.25,83,2.23,912.0
2018-12-04,2.2,2.0,2.25,78,2.27,892.0
2018-12-05,2.2,2.0,2.25,66,,
2018-12-06,2.2,2.0,2.25,71,2.34,963.0
2018-12-07,2.19,2.0,2.25,73,2.3,912.0
2018-12-10,2.2,2.0,2.25,70,2.22,903.0
2018-12-11,2.19,2.0,2.25,72,2.19,868.0
2018-12-12,2.19,2.0,2.25,74,2.2,882.0
2018-12-13,2.19,2.0,2.25,70,2.21,897.0
2018-12-14,2.19,2.0,2.25,64,2.21,867.0
2018-12-17,2.2,2.0,2.25,66,2.31,935.0
2018-12-18,2.2,2.0,2.25,61,2.32,920.0
2018-12-19,2.2,2.0,2.25,72,2.3,910.0
2018-12-20,2.4,2.25,2.5,50,2.41,963.0
2018-12-21,2.4,2.25,2.5,69,2.4,927.0
2018-12-24,2.4,2.25,2.5,69,2.41,881.0
2018-12-26,2.4,2.25,2.5,67,2.44,957.0
2018-12-27,2.4,2.25,2.5,71,2.44,943.0
2018-12-28,2.4,2.25,2.5,67,2.46,927.0
2018-12-31,2.4,2.25,2.5,55,3.0,1000.0
2019-01-02,2.4,2.25,2.5,56,3.15,1092.0
2019-01-03,2.4,2.25,2.5,62,2.7,1063.0
2019-01-04,2.4,2.25,2.5,68,2.45,1041.0
2019-01-07,2.4,2.25,2.5,68,2.41,1052.0
2019-01-08,2.4,2.25,2.5,70,2.42,1070.0
2019-01-09,2.4,2.25,2.5,64,2.45,1032.0
2019-01-10,2.4,2.25,2.5,65,2.43,1026.0
2019-01-11,2.4,2.25,2.5,63,2.41,973.0
2019-01-14,2.4,2.25,2.5,59,2.4,977.0
2019-01-15,2.4,2.25,2.5,58,2.46,991.0
2019-01-16,2.4,2.25,2.5,63,2.43,988.0
2019-01-17,2.4,2.25,2.5,63,2.41,969.0
2019-01-18,2.4,2.25,2.5,58,2.42,974.0
2019-01-22,2.4,2.25,2.5,48,2.41,967.0
2019-01-23,2.4,2.25,2.5,55,2.4,956.0
2019-01-24,2.4,2.25,2.5,64,2.41,994.0
2019-01-25,2.4,2.25,2.5,69,2.4,915.0
2019-01-28,2.4,2.25,2.5,68,2.39,981.0
2019-01-29,2.4,2.25,2.5,60,2.4,934.0
2019-01-30,2.4,2.25,2.5,63,2.39,948.0
2019-01-31,2.4,2.25,2.5,66,2.58,1048.0
2019-02-01,2.4,2.25,2.5,76,2.47,993.0
2019-02-04,2.4,2.25,2.5,82,2.4,964.0
2019-02-05,2.4,2.25,2.5,83,2.4,932.0
2019-02-06,2.4,2.25,2.5,83,2.38,942.0
2019-02-07,2.4,2.25,2.5,78,2.38,918.0
2019-02-08,2.4,2.25,2.5,77,2.37,912.0
2019-02-11,2.4,2.25,2.5,65,2.4,984.0
2019-02-12,2.4,2.25,2.5,65,2.39,920.0
2019-02-13,2.4,2.25,2.5,65,2.38,928.0
2019-02-14,2.4,2.25,2.5,68,2.39,862.0
2019-02-15,2.4,2.25,2.5,56,2.43,913.0
2019-02-19,2.4,2.25,2.5,61,2.4,941.0
2019-02-20,2.4,2.25,2.5,62,2.39,913.0
2019-02-21,2.4,2.25,2.5,74,2.39,886.0
2019-02-22,2.4,2.25,2.5,77,2.4,894.0
2019-02-25,2.4,2.25,2.5,72,2.38,938.0
2019-02-26,2.4,2.25,2.5,71,2.38,919.0
2019-02-27,2.4,2.25,2.5,80,2.37,898.0
2019-02-28,2.4,2.25,2.5,75,2.58,1003.0
2019-03-01,2.4,2.25,2.5,93,2.38,1007.0
2019-03-04,2.4,2.25,2.5,87,2.38,965.0
2019-03-05,2.4,2.25,2.5,88,2.38,956.0
2019-03-06,2.4,2.25,2.5,83,2.38,977.0
2019-03-07,2.4,2.25,2.5,79,2.38,972.0
2019-03-08,2.4,2.25,2.5,81,2.39,954.0
2019-03-11,2.4,2.25,2.5,79,2.38,949.0
2019-03-12,2.4,2.25,2.5,75,2.39,916.0
2019-03-13,2.4,2.25,2.5,74,2.4,925.0
2019-03-14,2.4,2.25,2.5,78,2.42,879.0
2019-03-15,2.4,2.25,2.5,69,2.46,940.0
2019-03-18,2.4,2.25,2.5,63,2.41,942.0
2019-03-19,2.4,2.25,2.5,61,2.42,945.0
2019-03-20,2.41,2.25,2.5,73,2.47,928.0
2019-03-21,2.41,2.25,2.5,69,2.44,921.0
2019-03-22,2.41,2.25,2.5,70,2.4,889.0
2019-03-25,2.4,2.25,2.5,78,2.4,919.0
2019-03-26,2.4,2.25,2.5,76,2.4,901.0
2019-03-27,2.41,2.25,2.5,72,2.4,879.0
2019-03-28,2.41,2.25,2.5,74,2.43,898.0
2019-03-29,2.43,2.25,2.5,71,2.65,897.0
2019-04-01,2.41,2.25,2.5,66,2.46,978.0
2019-04-02,2.41,2.25,2.5,78,2.46,978.0
2019-04-03,2.41,2.25,2.5,79,2.47,987.0
2019-04-04,2.41,2.25,2.5,79,2.46,1000.0
2019-04-05,2.41,2.25,2.5,73,2.46,976.0
2019-04-08,2.41,2.25,2.5,72,2.46,949.0
2019-04-09,2.41,2.25,2.5,72,2.45,970.0
2019-04-10,2.41,2.25,2.5,72,2.45,945.0
2019-04-11,2.41,2.25,2.5,73,2.44,974.0
2019-04-12,2.41,2.25,2.5,67,2.44,955.0
2019-04-15,2.41,2.25,2.5,57,2.47,968.0
2019-04-16,2.41,2.25,2.5,55,2.47,961.0
2019-04-17,2.42,2.25,2.5,55,2.5,954.0
2019-04-18,2.43,2.25,2.5,58,2.5,979.0
2019-04-19,2.44,2.25,2.5,59,,
2019-04-22,2.44,2.25,2.5,54,2.46,978.0
2019-04-23,2.44,2.25,2.5,53,2.46,969.0
2019-04-24,2.44,2.25,2.5,59,2.44,1018.0
2019-04-25,2.44,2.25,2.5,65,2.45,962.0
2019-04-26,2.44,2.25,2.5,73,2.46,982.0
2019-04-29,2.45,2.25,2.5,60,2.48,968.0
2019-04-30,2.45,2.25,2.5,68,2.76,1084.0
2019-05-01,2.45,2.25,2.5,77,2.54,1074.0
2019-05-02,2.41,2.25,2.5,69,2.5,1053.0
2019-05-03,2.4,2.25,2.5,74,2.43,1076.0
2019-05-06,2.4,2.25,2.5,69,2.42,1021.0
2019-05-07,2.4,2.25,2.5,64,2.44,1069.0
2019-05-08,2.39,2.25,2.5,70,2.43,1034.0
2019-05-09,2.38,2.25,2.5,73,2.41,1011.0
2019-05-10,2.38,2.25,2.5,71,2.4,1034.0
2019-05-13,2.38,2.25,2.5,67,2.38,1007.0
2019-05-14,2.38,2.25,2.5,64,2.39,1012.0
2019-05-15,2.4,2.25,2.5,62,2.48,1052.0
2019-05-16,2.39,2.25,2.5,61,2.43,1030.0
2019-05-17,2.39,2.25,2.5,67,2.42,1036.0
2019-05-20,2.39,2.25,2.5,56,2.39,1038.0
2019-05-21,2.39,2.25,2.5,64,2.38,1017.0
2019-05-22,2.38,2.25,2.5,67,2.37,1037.0
2019-05-23,2.38,2.25,2.5,71,2.37,1039.0
2019-05-24,2.38,2.25,2.5,66,2.37,1038.0
2019-05-28,2.39,2.25,2.5,64,2.41,1085.0
2019-05-29,2.39,2.25,2.5,58,2.4,1099.0
2019-05-30,2.39,2.25,2.5,60,2.4,1083.0
2019-05-31,2.4,2.25,2.5,65,2.49,1136.0
2019-06-03,2.38,2.25,2.5,76,2.4,1162.0
2019-06-04,2.38,2.25,2.5,71,2.39,1110.0
2019-06-05,2.38,2.25,2.5,66,2.4,1082.0
2019-06-06,2.37,2.25,2.5,71,2.4,1031.0
2019-06-07,2.37,2.25,2.5,68,2.39,1104.0
2019-06-10,2.37,2.25,2.5,62,2.39,1112.0
2019-06-11,2.37,2.25,2.5,66,2.38,1048.0
2019-06-12,2.37,2.25,2.5,69,2.37,1099.0
2019-06-13,2.37,2.25,2.5,64,2.35,1039.0
2019-06-14,2.36,2.25,2.5,63,2.35,1047.0
2019-06-17,2.38,2.25,2.5,55,2.41,1098.0
2019-06-18,2.37,2.25,2.5,59,2.39,1059.0
2019-06-19,2.37,2.25,2.5,60,2.36,1059.0
2019-06-20,2.37,2.25,2.5,61,2.36,1067.0
2019-06-21,2.38,2.25,2.5,65,2.37,1103.0
2019-06-24,2.38,2.25,2.5,63,2.39,1092.0
2019-06-25,2.38,2.25,2.5,70,2.41,1101.0
2019-06-26,2.38,2.25,2.5,70,2.43,1099.0
2019-06-27,2.38,2.25,2.5,76,2.42,1117.0
2019-06-28,2.4,2.25,2.5,73,2.5,1113.0
2019-07-01,2.39,2.25,2.5,73,2.42,1181.0
2019-07-02,2.4,2.25,2.5,71,2.51,1224.0
2019-07-03,2.41,2.25,2.5,75,2.56,1213.0
2019-07-05,2.42,2.25,2.5,72,2.59,1193.0
2019-07-08,2.41,2.25,2.5,76,2.48,1211.0
2019-07-09,2.41,2.25,2.5,75,2.45,1178.0
2019-07-10,2.41,2.25,2.5,78,2.46,1139.0
2019-07-11,2.4,2.25,2.5,77,2.41,1147.0
2019-07-12,2.38,2.25,2.5,69,2.36,1138.0
2019-07-15,2.4,2.25,2.5,63,2.46,1199.0
2019-07-16,2.41,2.25,2.5,64,2.47,1102.0
2019-07-17,2.41,2.25,2.5,62,2.47,1104.0
2019-07-18,2.41,2.25,2.5,55,2.46,1114.0
2019-07-19,2.41,2.25,2.5,58,2.41,1140.0
2019-07-22,2.4,2.25,2.5,60,2.4,1123.0
2019-07-23,2.4,2.25,2.5,60,2.4,1091.0
2019-07-24,2.4,2.25,2.5,65,2.41,1139.0
2019-07-25,2.4,2.25,2.5,69,2.42,1153.0
2019-07-26,2.4,2.25,2.5,78,2.41,1142.0
2019-07-29,2.4,2.25,2.5,77,2.4,1153.0
2019-07-30,2.39,2.25,2.5,72,2.39,1158.0
2019-07-31,2.4,2.25,2.5,70,2.55,1217.0
2019-08-01,2.14,2.0,2.25,81,2.19,1183.0
2019-08-02,2.14,2.0,2.25,74,2.19,1185.0
2019-08-05,2.13,2.0,2.25,68,2.13,1253.0
2019-08-06,2.13,2.0,2.25,65,2.11,1234.0
2019-08-07,2.12,2.0,2.25,66,2.1,1196.0
2019-08-08,2.12,2.0,2.25,62,2.09,1218.0
2019-08-09,2.12,2.0,2.25,69,2.11,1186.0
2019-08-12,2.12,2.0,2.25,68,2.12,1177.0
2019-08-13,2.12,2.0,2.25,66,2.16,1183.0
2019-08-14,2.12,2.0,2.25,60,2.13,1244.0
2019-08-15,2.13,2.0,2.25,61,2.18,1218.0
2019-08-16,2.13,2.0,2.25,62,2.13,1247.0
2019-08-19,2.13,2.0,2.25,56,2.11,1241.0
2019-08-20,2.13,2.0,2.25,55,2.13,1228.0
2019-08-21,2.12,2.0,2.25,65,2.1,1188.0
2019-08-22,2.12,2.0,2.25,65,2.09,1160.0
2019-08-23,2.12,2.0,2.25,68,2.1,1130.0
2019-08-26,2.12,2.0,2.25,64,2.1,1190.0
2019-08-27,2.12,2.0,2.25,62,2.15,1197.0
2019-08-28,2.12,2.0,2.25,62,2.12,1196.0
2019-08-29,2.12,2.0,2.25,62,2.12,1196.0
2019-08-30,2.13,2.0,2.25,66,2.16,1194.0
2019-09-03,2.13,2.0,2.25,71,2.17,1281.0
2019-09-04,2.13,2.0,2.25,72,2.21,1222.0
2019-09-05,2.13,2.0,2.25,65,2.21,1197.0
2019-09-06,2.12,2.0,2.25,69,2.15,1222.0
2019-09-09,2.13,2.0,2.25,64,2.12,1243.0
2019-09-10,2.13,2.0,2.25,63,2.14,1186.0
2019-09-11,2.13,2.0,2.25,61,2.15,1179.0
2019-09-12,2.13,2.0,2.25,60,2.2,1156.0
2019-09-13,2.14,2.0,2.25,53,2.2,1134.0
2019-09-16,2.25,2.0,2.25,46,2.43,1156.0
2019-09-17,2.3,2.0,2.25,61,5.25,1177.0
2019-09-18,2.25,2.0,2.25,58,2.55,1196.0
2019-09-19,1.9,1.75,2.0,59,1.95,1137.0
2019-09-20,1.9,1.75,2.0,65,1.86,1121.0
2019-09-23,1.9,1.75,2.0,74,1.85,1140.0
2019-09-24,1.9,1.75,2.0,71,1.96,1085.0
2019-09-25,1.9,1.75,2.0,77,2.01,1080.0
2019-09-26,1.85,1.75,2.0,89,1.85,1092.0
2019-09-27,1.83,1.75,2.0,85,1.82,1084.0
2019-09-30,1.9,1.75,2.0,59,2.35,1097.0
2019-10-01,1.88,1.75,2.0,75,1.88,1183.0
2019-10-02,1.85,1.75,2.0,76,1.85,1147.0
2019-10-03,1.83,1.75,2.0,80,1.84,1120.0
2019-10-04,1.82,1.75,2.0,80,1.82,1094.0
2019-10-07,1.82,1.75,2.0,76,1.83,1070.0
2019-10-08,1.82,1.75,2.0,69,1.85,1070.0
2019-10-09,1.82,1.75,2.0,66,1.85,1061.0
2019-10-10,1.82,1.75,2.0,61,1.85,1124.0
2019-10-11,1.82,1.75,2.0,60,1.85,1024.0
2019-10-15,1.9,1.75,2.0,69,2.0,1088.0
2019-10-16,1.9,1.75,2.0,76,2.05,1086.0
2019-10-17,1.85,1.75,2.0,66,1.95,1077.0
2019-10-18,1.85,1.75,2.0,61,1.88,1120.0
2019-10-21,1.85,1.75,2.0,60,1.86,1054.0
2019-10-22,1.85,1.75,2.0,67,1.87,1079.0
2019-10-23,1.85,1.75,2.0,65,1.87,1065.0
2019-10-24,1.85,1.75,2.0,65,1.86,1086.0
2019-10-25,1.83,1.75,2.0,73,1.84,1077.0
2019-10-28,1.83,1.75,2.0,72,1.82,1089.0
2019-10-29,1.82,1.75,2.0,72,1.81,1048.0
2019-10-30,1.82,1.75,2.0,69,1.82,1099.0
2019-10-31,1.58,1.5,1.75,71,1.76,1174.0
2019-11-01,1.57,1.5,1.75,83,1.58,1118.0
2019-11-04,1.56,1.5,1.75,83,1.56,1146.0
2019-11-05,1.56,1.5,1.75,81,1.58,1062.0
2019-11-06,1.55,1.5,1.75,80,1.57,1083.0
2019-11-07,1.55,1.5,1.75,76,1.56,1066.0
2019-11-08,1.55,1.5,1.75,71,1.56,1016.0
2019-11-12,1.55,1.5,1.75,71,1.57,1064.0
2019-11-13,1.55,1.5,1.75,71,1.57,1018.0
2019-11-14,1.55,1.5,1.75,68,1.58,1027.0
2019-11-15,1.55,1.5,1.75,73,1.59,1049.0
2019-11-18,1.55,1.5,1.75,62,1.56,1012.0
2019-11-19,1.55,1.5,1.75,64,1.57,1008.0
2019-11-20,1.55,1.5,1.75,70,1.57,1001.0
2019-11-21,1.55,1.5,1.75,67,1.58,993.0
2019-11-22,1.55,1.5,1.75,65,1.58,1028.0
2019-11-25,1.55,1.5,1.75,69,1.56,994.0
2019-11-26,1.55,1.5,1.75,71,1.54,1056.0
2019-11-27,1.55,1.5,1.75,68,1.55,1011.0
2019-11-29,1.56,1.5,1.75,70,1.65,1004.0
2019-12-02,1.56,1.5,1.75,72,1.63,1104.0
2019-12-03,1.55,1.5,1.75,76,1.55,1050.0
2019-12-04,1.55,1.5,1.75,75,1.54,1046.0
2019-12-05,1.55,1.5,1.75,70,1.55,1017.0
2019-12-06,1.55,1.5,1.75,82,1.55,1017.0
2019-12-09,1.55,1.5,1.75,73,1.56,1042.0
2019-12-10,1.55,1.5,1.75,75,1.55,1049.0
2019-12-11,1.55,1.5,1.75,73,1.54,1056.0
2019-12-12,1.55,1.5,1.75,75,1.53,1039.0
2019-12-13,1.55,1.5,1.75,70,1.54,1017.0
2019-12-16,1.56,1.5,1.75,69,1.62,1025.0
2019-12-17,1.55,1.5,1.75,68,1.54,1014.0
2019-12-18,1.55,1.5,1.75,74,1.53,1057.0
2019-12-19,1.55,1.5,1.75,73,1.53,988.0
2019-12-20,1.55,1.5,1.75,69,1.53,967.0
2019-12-23,1.55,1.5,1.75,69,1.52,1027.0
2019-12-24,1.55,1.5,1.75,71,1.52,1006.0
2019-12-26,1.55,1.5,1.75,75,1.52,1000.0
2019-12-27,1.55,1.5,1.75,73,1.53,1038.0
2019-12-30,1.55,1.5,1.75,72,1.54,1036.0
2019-12-31,1.55,1.5,1.75,51,1.55,1010.0
2020-01-02,1.55,1.5,1.75,74,1.54,1123.0
2020-01-03,1.55,1.5,1.75,79,1.55,1030.0
2020-01-06,1.55,1.5,1.75,77,1.55,1034.0
2020-01-07,1.55,1.5,1.75,80,1.56,1028.0
2020-01-08,1.55,1.5,1.75,83,1.55,1007.0
2020-01-09,1.55,1.5,1.75,81,1.55,1011.0
2020-01-10,1.54,1.5,1.75,83,1.55,1013.0
2020-01-13,1.54,1.5,1.75,85,1.54,1021.0
2020-01-14,1.54,1.5,1.75,76,1.55,1009.0
2020-01-15,1.54,1.5,1.75,76,1.56,1022.0
2020-01-16,1.54,1.5,1.75,78,1.55,1069.0
2020-01-17,1.55,1.5,1.75,67,1.54,1043.0
2020-01-21,1.55,1.5,1.75,64,1.54,1067.0
2020-01-22,1.55,1.5,1.75,69,1.54,1085.0
2020-01-23,1.55,1.5,1.75,67,1.54,1051.0
2020-01-24,1.55,1.5,1.75,70,1.53,1070.0
2020-01-27,1.55,1.5,1.75,71,1.53,1051.0
2020-01-28,1.55,1.5,1.75,69,1.53,1074.0
2020-01-29,1.55,1.5,1.75,71,1.53,1082.0
2020-01-30,1.6,1.5,1.75,64,1.58,1072.0
2020-01-31,1.59,1.5,1.75,78,1.6,1149.0
2020-02-03,1.59,1.5,1.75,79,1.59,1134.0
2020-02-04,1.59,1.5,1.75,80,1.6,1161.0
2020-02-05,1.59,1.5,1.75,83,1.59,1168.0
2020-02-06,1.59,1.5,1.75,79,1.59,1105.0
2020-02-07,1.58,1.5,1.75,77,1.58,1111.0
2020-02-10,1.58,1.5,1.75,69,1.58,1111.0
2020-02-11,1.58,1.5,1.75,66,1.58,1114.0
2020-02-12,1.58,1.5,1.75,74,1.57,1124.0
2020-02-13,1.58,1.5,1.75,69,1.57,1114.0
2020-02-14,1.58,1.5,1.75,61,1.58,1086.0
2020-02-18,1.59,1.5,1.75,61,1.6,1163.0
2020-02-19,1.59,1.5,1.75,64,1.59,1128.0
2020-02-20,1.59,1.5,1.75,60,1.6,1112.0
2020-02-21,1.58,1.5,1.75,65,1.58,1099.0
2020-02-24,1.58,1.5,1.75,63,1.58,1106.0
2020-02-25,1.58,1.5,1.75,64,1.59,1122.0
2020-02-26,1.58,1.5,1.75,62,1.58,1120.0
2020-02-27,1.58,1.5,1.75,65,1.58,1106.0
2020-02-28,1.58,1.5,1.75,61,1.6,1156.0
2020-03-02,1.59,1.5,1.75,58,1.59,1291.0
2020-03-03,1.59,1.5,1.75,58,1.64,1238.0
2020-03-04,1.09,1.0,1.25,80,1.23,1261.0
2020-03-05,1.09,1.0,1.25,78,1.12,1254.0
2020-03-06,1.09,1.0,1.25,79,1.1,1295.0
2020-03-09,1.09,1.0,1.25,77,1.09,1256.0
2020-03-10,1.09,1.0,1.25,77,1.11,1264.0
2020-03-11,1.09,1.0,1.25,80,1.15,1222.0
2020-03-12,1.1,1.0,1.25,61,1.2,1293.0
2020-03-13,1.1,1.0,1.25,65,1.1,1298.0
2020-03-16,0.25,0.0,0.25,80,0.26,1306.0
2020-03-17,0.25,0.0,0.25,85,0.54,1335.0
2020-03-18,0.25,0.0,0.25,82,0.1,1358.0
2020-03-19,0.2,0.0,0.25,103,0.06,1319.0
2020-03-20,0.15,0.0,0.25,87,0.04,1235.0
2020-03-23,0.15,0.0,0.25,82,0.02,1277.0
2020-03-24,0.12,0.0,0.25,95,0.01,1280.0
2020-03-25,0.1,0.0,0.25,102,0.01,1229.0
2020-03-26,0.1,0.0,0.25,101,0.01,1233.0
2020-03-27,0.1,0.0,0.25,94,0.01,1222.0
2020-03-30,0.09,0.0,0.25,84,0.01,1157.0
2020-03-31,0.08,0.0,0.25,78,0.01,1127.0
2020-04-01,0.06,0.0,0.25,92,0.01,1226.0
2020-04-02,0.05,0.0,0.25,90,0.01,1202.0
2020-04-03,0.05,0.0,0.25,87,0.01,1266.0
2020-04-06,0.05,0.0,0.25,84,0.01,1254.0
2020-04-07,0.05,0.0,0.25,85,0.01,1226.0
2020-04-08,0.05,0.0,0.25,85,0.01,1152.0
2020-04-09,0.05,0.0,0.25,83,0.01,1160.0
2020-04-10,0.05,0.0,0.25,88,,
2020-04-13,0.05,0.0,0.25,95,0.02,1193.0
2020-04-14,0.05,0.0,0.25,92,0.06,1152.0
2020-04-15,0.05,0.0,0.25,101,0.03,1257.0
2020-04-16,0.05,0.0,0.25,99,0.03,1223.0
2020-04-17,0.05,0.0,0.25,101,0.03,1231.0
2020-04-20,0.05,0.0,0.25,106,0.02,1158.0
2020-04-21,0.05,0.0,0.25,104,0.01,1163.0
2020-04-22,0.05,0.0,0.25,107,0.01,1158.0
2020-04-23,0.04,0.0,0.25,107,0.01,1141.0
2020-04-24,0.05,0.0,0.25,99,0.03,1208.0
2020-04-27,0.04,0.0,0.25,100,0.03,1145.0
2020-04-28,0.04,0.0,0.25,97,0.01,1165.0
2020-04-29,0.04,0.0,0.25,99,0.01,1141.0
2020-04-30,0.05,0.0,0.25,92,0.04,1180.0
2020-05-01,0.05,0.0,0.25,92,0.03,1190.0
2020-05-04,0.05,0.0,0.25,90,0.05,1184.0
2020-05-05,0.05,0.0,0.25,92,0.05,1172.0
2020-05-06,0.05,0.0,0.25,90,0.05,1127.0
2020-05-07,0.05,0.0,0.25,92,0.05,1127.0
2020-05-08,0.05,0.0,0.25,85,0.06,1142.0
2020-05-11,0.05,0.0,0.25,84,0.06,1123.0
2020-05-12,0.05,0.0,0.25,88,0.06,1142.0
2020-05-13,0.05,0.0,0.25,90,0.04,1085.0
2020-05-14,0.05,0.0,0.25,92,0.02,1102.0
2020-05-15,0.05,0.0,0.25,93,0.05,1127.0
2020-05-18,0.05,0.0,0.25,87,0.04,1130.0
2020-05-19,0.05,0.0,0.25,86,0.04,1127.0
2020-05-20,0.05,0.0,0.25,86,0.01,1103.0
2020-05-21,0.05,0.0,0.25,85,0.02,1117.0
2020-05-22,0.05,0.0,0.25,78,0.04,1085.0
2020-05-26,0.05,0.0,0.25,76,0.06,1081.0
2020-05-27,0.05,0.0,0.25,79,0.06,1069.0
2020-05-28,0.05,0.0,0.25,70,0.06,1031.0
2020-05-29,0.05,0.0,0.25,65,0.06,1061.0
2020-06-01,0.05,0.0,0.25,66,0.06,1104.0
2020-06-02,0.06,0.0,0.25,68,0.07,1054.0
2020-06-03,0.06,0.0,0.25,62,0.06,1024.0
2020-06-04,0.06,0.0,0.25,61,0.07,1067.0
2020-06-05,0.07,0.0,0.25,57,0.07,1021.0
2020-06-08,0.07,0.0,0.25,60,0.07,1032.0
2020-06-09,0.07,0.0,0.25,59,0.08,1034.0
2020-06-10,0.08,0.0,0.25,53,0.07,1008.0
2020-06-11,0.08,0.0,0.25,55,0.08,1009.0
2020-06-12,0.08,0.0,0.25,52,0.08,969.0
2020-06-15,0.09,0.0,0.25,55,0.09,1000.0
2020-06-16,0.09,0.0,0.25,57,0.09,986.0
2020-06-17,0.09,0.0,0.25,59,0.09,1012.0
2020-06-18,0.09,0.0,0.25,65,0.09,1046.0
2020-06-19,0.09,0.0,0.25,73,0.09,963.0
2020-06-22,0.08,0.0,0.25,76,0.08,1024.0
2020-06-23,0.08,0.0,0.25,71,0.07,992.0
2020-06-24,0.08,0.0,0.25,74,0.08,1015.0
2020-06-25,0.08,0.0,0.25,79,0.09,984.0
2020-06-26,0.08,0.0,0.25,78,0.08,938.0
2020-06-29,0.08,0.0,0.25,71,0.08,940.0
2020-06-30,0.08,0.0,0.25,58,0.1,970.0
2020-07-01,0.08,0.0,0.25,66,0.11,1003.0
2020-07-02,0.09,0.0,0.25,65,0.11,952.0
2020-07-03,0.09,0.0,0.25,65,,
2020-07-06,0.09,0.0,0.25,62,0.1,977.0
2020-07-07,0.09,0.0,0.25,59,0.1,969.0
2020-07-08,0.09,0.0,0.25,59,0.1,954.0
2020-07-09,0.09,0.0,0.25,60,0.1,984.0
2020-07-10,0.09,0.0,0.25,56,0.1,942.0
2020-07-13,0.09,0.0,0.25,54,0.1,960.0
2020-07-14,0.09,0.0,0.25,54,0.11,972.0
2020-07-15,0.1,0.0,0.25,55,0.13,939.0
2020-07-16,0.1,0.0,0.25,55,0.12,957.0
2020-07-17,0.09,0.0,0.25,49,0.12,988.0
2020-07-20,0.1,0.0,0.25,43,0.12,975.0
2020-07-21,0.1,0.0,0.25,44,0.11,963.0
2020-07-22,0.09,0.0,0.25,47,0.1,1031.0
2020-07-23,0.09,0.0,0.25,50,0.1,942.0
2020-07-24,0.09,0.0,0.25,45,0.1,950.0
2020-07-27,0.1,0.0,0.25,46,0.1,973.0
2020-07-28,0.1,0.0,0.25,47,0.1,955.0
2020-07-29,0.1,0.0,0.25,46,0.09,948.0
2020-07-30,0.1,0.0,0.25,56,0.1,928.0
2020-07-31,0.1,0.0,0.25,50,0.1,1034.0
2020-08-03,0.1,0.0,0.25,51,0.1,991.0
2020-08-04,0.1,0.0,0.25,53,0.09,1051.0
2020-08-05,0.1,0.0,0.25,53,0.09,1028.0
2020-08-06,0.1,0.0,0.25,56,0.09,1029.0
2020-08-07,0.1,0.0,0.25,54,0.09,944.0
2020-08-10,0.1,0.0,0.25,51,0.09,948.0
2020-08-11,0.1,0.0,0.25,54,0.1,956.0
2020-08-12,0.1,0.0,0.25,55,0.09,964.0
2020-08-13,0.1,0.0,0.25,56,0.09,989.0
2020-08-14,0.1,0.0,0.25,56,0.09,930.0
2020-08-17,0.1,0.0,0.25,57,0.1,944.0
2020-08-18,0.09,0.0,0.25,57,0.09,977.0
2020-08-19,0.09,0.0,0.25,58,0.09,961.0
2020-08-20,0.09,0.0,0.25,59,0.07,983.0
2020-08-21,0.09,0.0,0.25,59,0.07,959.0
2020-08-24,0.09,0.0,0.25,58,0.08,935.0
2020-08-25,0.09,0.0,0.25,66,0.08,931.0
2020-08-26,0.09,0.0,0.25,65,0.07,938.0
2020-08-27,0.08,0.0,0.25,69,0.07,898.0
2020-08-28,0.09,0.0,0.25,66,0.07,881.0
2020-08-31,0.09,0.0,0.25,63,0.09,934.0
2020-09-01,0.09,0.0,0.25,62,0.09,969.0
2020-09-02,0.09,0.0,0.25,63,0.09,947.0
2020-09-03,0.09,0.0,0.25,63,0.1,955.0
2020-09-04,0.09,0.0,0.25,59,0.09,935.0
2020-09-08,0.09,0.0,0.25,63,0.09,957.0
2020-09-09,0.09,0.0,0.25,59,0.09,937.0
2020-09-10,0.09,0.0,0.25,64,0.09,924.0
2020-09-11,0.09,0.0,0.25,52,0.09,900.0
2020-09-14,0.09,0.0,0.25,57,0.09,881.0
2020-09-15,0.09,0.0,0.25,55,0.1,938.0
2020-09-16,0.09,0.0,0.25,61,0.1,899.0
2020-09-17,0.09,0.0,0.25,73,0.1,892.0
2020-09-18,0.09,0.0,0.25,68,0.09,896.0
2020-09-21,0.09,0.0,0.25,64,0.08,925.0
2020-09-22,0.09,0.0,0.25,66,0.07,941.0
2020-09-23,0.09,0.0,0.25,70,0.06,945.0
2020-09-24,0.09,0.0,0.25,72,0.07,919.0
2020-09-25,0.09,0.0,0.25,68,0.08,929.0
2020-09-28,0.09,0.0,0.25,70,0.08,923.0
2020-09-29,0.09,0.0,0.25,64,0.07,916.0
2020-09-30,0.09,0.0,0.25,47,0.08,975.0
2020-10-01,0.09,0.0,0.25,61,0.08,1020.0
2020-10-02,0.09,0.0,0.25,55,0.1,927.0
2020-10-05,0.09,0.0,0.25,57,0.09,955.0
2020-10-06,0.09,0.0,0.25,61,0.1,920.0
2020-10-07,0.09,0.0,0.25,58,0.09,925.0
2020-10-08,0.09,0.0,0.25,56,0.08,924.0
2020-10-09,0.09,0.0,0.25,52,0.09,894.0
2020-10-13,0.09,0.0,0.25,54,0.09,930.0
2020-10-14,0.09,0.0,0.25,54,0.09,885.0
2020-10-15,0.09,0.0,0.25,58,0.1,952.0
2020-10-16,0.09,0.0,0.25,59,0.09,914.0
2020-10-19,0.09,0.0,0.25,54,0.09,903.0
2020-10-20,0.09,0.0,0.25,54,0.08,918.0
2020-10-21,0.09,0.0,0.25,59,0.07,943.0
2020-10-22,0.09,0.0,0.25,61,0.07,929.0
2020-10-23,0.09,0.0,0.25,57,0.08,915.0
2020-10-26,0.09,0.0,0.25,58,0.09,921.0
2020-10-27,0.09,0.0,0.25,60,0.09,931.0
2020-10-28,0.09,0.0,0.25,59,0.08,910.0
2020-10-29,0.09,0.0,0.25,59,0.09,866.0
2020-10-30,0.09,0.0,0.25,56,0.09,873.0
2020-11-02,0.09,0.0,0.25,58,0.11,956.0
2020-11-03,0.09,0.0,0.25,64,0.1,948.0
2020-11-04,0.09,0.0,0.25,67,0.1,899.0
2020-11-05,0.09,0.0,0.25,65,0.11,938.0
2020-11-06,0.09,0.0,0.25,61,0.1,888.0
2020-11-09,0.09,0.0,0.25,57,0.1,904.0
2020-11-10,0.09,0.0,0.25,57,0.1,923.0
2020-11-12,0.09,0.0,0.25,57,0.09,912.0
2020-11-13,0.09,0.0,0.25,55,0.09,904.0
2020-11-16,0.09,0.0,0.25,58,0.1,953.0
2020-11-17,0.09,0.0,0.25,60,0.09,970.0
2020-11-18,0.09,0.0,0.25,60,0.07,941.0
2020-11-19,0.08,0.0,0.25,62,0.06,911.0
2020-11-20,0.08,0.0,0.25,64,0.05,920.0
2020-11-23,0.08,0.0,0.25,62,0.05,913.0
2020-11-24,0.08,0.0,0.25,62,0.07,910.0
2020-11-25,0.08,0.0,0.25,66,0.08,937.0
2020-11-27,0.08,0.0,0.25,61,0.08,900.0
2020-11-30,0.09,0.0,0.25,57,0.09,1055.0
2020-12-01,0.09,0.0,0.25,56,0.08,972.0
2020-12-02,0.09,0.0,0.25,55,0.08,926.0
2020-12-03,0.09,0.0,0.25,60,0.08,954.0
2020-12-04,0.09,0.0,0.25,56,0.09,951.0
2020-12-07,0.09,0.0,0.25,57,0.08,951.0
2020-12-08,0.09,0.0,0.25,57,0.07,939.0
2020-12-09,0.09,0.0,0.25,57,0.08,935.0
2020-12-10,0.09,0.0,0.25,57,0.08,946.0
2020-12-11,0.09,0.0,0.25,51,0.08,885.0
2020-12-14,0.09,0.0,0.25,51,0.08,907.0
2020-12-15,0.09,0.0,0.25,50,0.09,933.0
2020-12-16,0.09,0.0,0.25,57,0.09,914.0
2020-12-17,0.09,0.0,0.25,58,0.09,922.0
2020-12-18,0.09,0.0,0.25,57,0.09,943.0
2020-12-21,0.09,0.0,0.25,66,0.09,922.0
2020-12-22,0.09,0.0,0.25,66,0.07,923.0
2020-12-23,0.09,0.0,0.25,70,0.06,921.0
2020-12-24,0.09,0.0,0.25,70,0.08,895.0
2020-12-28,0.09,0.0,0.25,73,0.09,946.0
2020-12-29,0.09,0.0,0.25,69,0.1,945.0
2020-12-30,0.09,0.0,0.25,62,0.09,916.0
2020-12-31,0.09,0.0,0.25,34,0.07,1095.0
2021-01-04,0.09,0.0,0.25,57,0.1,1046.0
2021-01-05,0.09,0.0,0.25,55,0.11,986.0
2021-01-06,0.09,0.0,0.25,56,0.1,978.0
2021-01-07,0.09,0.0,0.25,59,0.1,951.0
2021-01-08,0.09,0.0,0.25,55,0.09,992.0
2021-01-11,0.09,0.0,0.25,55,0.09,964.0
2021-01-12,0.09,0.0,0.25,61,0.08,918.0
2021-01-13,0.09,0.0,0.25,64,0.08,911.0
2021-01-14,0.09,0.0,0.25,64,0.08,887.0
2021-01-15,0.09,0.0,0.25,64,0.08,904.0
2021-01-19,0.09,0.0,0.25,61,0.07,936.0
2021-01-20,0.09,0.0,0.25,72,0.06,883.0
2021-01-21,0.08,0.0,0.25,74,0.04,922.0
2021-01-22,0.08,0.0,0.25,70,0.05,906.0
2021-01-25,0.08,0.0,0.25,74,0.06,885.0
2021-01-26,0.08,0.0,0.25,74,0.03,926.0
2021-01-27,0.08,0.0,0.25,71,0.03,913.0
2021-01-28,0.07,0.0,0.25,67,0.04,906.0
2021-01-29,0.07,0.0,0.25,62,0.06,915.0
2021-02-01,0.08,0.0,0.25,64,0.07,1010.0
2021-02-02,0.08,0.0,0.25,67,0.07,968.0
2021-02-03,0.08,0.0,0.25,67,0.07,962.0
2021-02-04,0.08,0.0,0.25,69,0.05,989.0
2021-02-05,0.08,0.0,0.25,72,0.02,933.0
2021-02-08,0.07,0.0,0.25,72,0.02,920.0
2021-02-09,0.08,0.0,0.25,73,0.05,901.0
2021-02-10,0.08,0.0,0.25,68,0.06,884.0
2021-02-11,0.08,0.0,0.25,67,0.06,891.0
2021-02-12,0.08,0.0,0.25,63,0.05,890.0
2021-02-16,0.08,0.0,0.25,65,0.06,934.0
2021-02-17,0.08,0.0,0.25,66,0.06,957.0
2021-02-18,0.07,0.0,0.25,73,0.03,872.0
2021-02-19,0.07,0.0,0.25,71,0.02,873.0
2021-02-22,0.07,0.0,0.25,70,0.03,860.0
2021-02-23,0.07,0.0,0.25,70,0.01,917.0
2021-02-24,0.07,0.0,0.25,70,0.02,870.0
2021-02-25,0.07,0.0,0.25,77,0.03,896.0
2021-02-26,0.07,0.0,0.25,71,0.01,889.0
2021-03-01,0.07,0.0,0.25,68,0.02,988.0
2021-03-02,0.07,0.0,0.25,68,0.04,951.0
2021-03-03,0.07,0.0,0.25,68,0.04,897.0
2021-03-04,0.07,0.0,0.25,69,0.02,936.0
2021-03-05,0.07,0.0,0.25,69,0.02,933.0
2021-03-08,0.07,0.0,0.25,66,0.02,928.0
2021-03-09,0.07,0.0,0.25,69,0.02,914.0
2021-03-10,0.07,0.0,0.25,66,0.02,914.0
2021-03-11,0.07,0.0,0.25,67,0.01,884.0
2021-03-12,0.07,0.0,0.25,63,0.01,889.0
2021-03-15,0.07,0.0,0.25,70,0.01,898.0
2021-03-16,0.07,0.0,0.25,69,0.01,914.0
2021-03-17,0.07,0.0,0.25,72,0.01,883.0
2021-03-18,0.07,0.0,0.25,75,0.01,868.0
2021-03-19,0.07,0.0,0.25,72,0.01,865.0
2021-03-22,0.07,0.0,0.25,68,0.01,869.0
2021-03-23,0.07,0.0,0.25,71,0.01,896.0
2021-03-24,0.07,0.0,0.25,70,0.01,895.0
2021-03-25,0.07,0.0,0.25,79,0.01,956.0
2021-03-26,0.07,0.0,0.25,75,0.01,924.0
2021-03-29,0.07,0.0,0.25,74,0.01,858.0
2021-03-30,0.07,0.0,0.25,76,0.01,808.0
2021-03-31,0.06,0.0,0.25,56,0.01,931.0
2021-04-01,0.07,0.0,0.25,72,0.01,989.0
2021-04-02,0.07,0.0,0.25,72,,
2021-04-05,0.07,0.0,0.25,73,0.01,985.0
2021-04-06,0.07,0.0,0.25,74,0.01,979.0
2021-04-07,0.07,0.0,0.25,73,0.01,924.0
2021-04-08,0.07,0.0,0.25,75,0.01,896.0
2021-04-09,0.07,0.0,0.25,69,0.01,867.0
2021-04-12,0.07,0.0,0.25,68,0.01,863.0
2021-04-13,0.07,0.0,0.25,67,0.01,894.0
2021-04-14,0.07,0.0,0.25,68,0.01,883.0
2021-04-15,0.07,0.0,0.25,66,0.01,928.0
2021-04-16,0.07,0.0,0.25,66,0.01,913.0
2021-04-19,0.07,0.0,0.25,65,0.01,919.0
2021-04-20,0.07,0.0,0.25,70,0.01,896.0
2021-04-21,0.07,0.0,0.25,72,0.01,882.0
2021-04-22,0.07,0.0,0.25,70,0.01,870.0
2021-04-23,0.07,0.0,0.25,66,0.01,871.0
2021-04-26,0.07,0.0,0.25,72,0.01,875.0
2021-04-27,0.07,0.0,0.25,65,0.01,879.0
2021-04-28,0.07,0.0,0.25,67,0.01,847.0
2021-04-29,0.06,0.0,0.25,72,0.01,838.0
2021-04-30,0.05,0.0,0.25,70,0.01,924.0
2021-05-03,0.06,0.0,0.25,74,0.01,930.0
2021-05-04,0.06,0.0,0.25,73,0.01,895.0
2021-05-05,0.06,0.0,0.25,73,0.01,841.0
2021-05-06,0.06,0.0,0.25,72,0.01,883.0
2021-05-07,0.06,0.0,0.25,69,0.01,869.0
2021-05-10,0.06,0.0,0.25,66,0.01,869.0
2021-05-11,0.06,0.0,0.25,64,0.01,865.0
2021-05-12,0.06,0.0,0.25,67,0.01,895.0
2021-05-13,0.06,0.0,0.25,63,0.01,860.0
2021-05-14,0.06,0.0,0.25,62,0.01,865.0
2021-05-17,0.06,0.0,0.25,63,0.01,902.0
2021-05-18,0.06,0.0,0.25,63,0.01,888.0
2021-05-19,0.06,0.0,0.25,60,0.01,891.0
2021-05-20,0.06,0.0,0.25,66,0.01,872.0
2021-05-21,0.06,0.0,0.25,65,0.01,857.0
2021-05-24,0.06,0.0,0.25,64,0.01,858.0
2021-05-25,0.06,0.0,0.25,65,0.01,877.0
2021-05-26,0.06,0.0,0.25,64,0.01,869.0
2021-05-27,0.06,0.0,0.25,67,0.01,877.0
2021-05-28,0.05,0.0,0.25,61,0.01,868.0
2021-06-01,0.06,0.0,0.25,60,0.01,977.0
2021-06-02,0.06,0.0,0.25,70,0.01,953.0
2021-06-03,0.06,0.0,0.25,69,0.01,903.0
2021-06-04,0.06,0.0,0.25,68,0.01,898.0
2021-06-07,0.06,0.0,0.25,67,0.01,921.0
2021-06-08,0.06,0.0,0.25,64,0.01,896.0
2021-06-09,0.06,0.0,0.25,64,0.01,896.0
2021-06-10,0.06,0.0,0.25,63,0.01,904.0
2021-06-11,0.06,0.0,0.25,60,0.01,834.0
2021-06-14,0.06,0.0,0.25,58,0.01,877.0
2021-06-15,0.06,0.0,0.25,60,0.01,905.0
2021-06-16,0.06,0.0,0.25,62,0.01,900.0
2021-06-17,0.1,0.0,0.25,64,0.05,998.0
2021-06-18,0.1,0.0,0.25,68,0.05,921.0
2021-06-21,0.1,0.0,0.25,66,0.05,894.0
2021-06-22,0.1,0.0,0.25,70,0.05,888.0
2021-06-23,0.1,0.0,0.25,71,0.05,835.0
2021-06-24,0.1,0.0,0.25,74,0.05,868.0
2021-06-25,0.1,0.0,0.25,77,0.05,850.0
2021-06-28,0.1,0.0,0.25,77,0.05,853.0
2021-06-29,0.1,0.0,0.25,75,0.05,850.0
2021-06-30,0.08,0.0,0.25,49,0.05,871.0
2021-07-01,0.1,0.0,0.25,70,0.05,938.0
2021-07-02,0.1,0.0,0.25,81,0.05,910.0
2021-07-06,0.1,0.0,0.25,76,0.05,934.0
2021-07-07,0.1,0.0,0.25,77,0.05,905.0
2021-07-08,0.1,0.0,0.25,76,0.05,874.0
2021-07-09,0.1,0.0,0.25,75,0.05,911.0
2021-07-12,0.1,0.0,0.25,75,0.05,855.0
2021-07-13,0.1,0.0,0.25,74,0.05,875.0
2021-07-14,0.1,0.0,0.25,72,0.05,908.0
2021-07-15,0.1,0.0,0.25,73,0.05,933.0
2021-07-16,0.1,0.0,0.25,72,0.05,864.0
2021-07-19,0.1,0.0,0.25,67,0.05,888.0
2021-07-20,0.1,0.0,0.25,70,0.05,855.0
2021-07-21,0.1,0.0,0.25,71,0.05,913.0
2021-07-22,0.1,0.0,0.25,71,0.05,849.0
2021-07-23,0.1,0.0,0.25,68,0.05,861.0
2021-07-26,0.1,0.0,0.25,73,0.05,881.0
2021-07-27,0.1,0.0,0.25,69,0.05,874.0
2021-07-28,0.1,0.0,0.25,65,0.05,862.0
2021-07-29,0.1,0.0,0.25,70,0.05,877.0
2021-07-30,0.07,0.0,0.25,61,0.05,903.0
2021-08-02,0.1,0.0,0.25,74,0.05,986.0
2021-08-03,0.1,0.0,0.25,69,0.05,974.0
2021-08-04,0.1,0.0,0.25,71,0.05,926.0
2021-08-05,0.1,0.0,0.25,70,0.05,901.0
2021-08-06,0.1,0.0,0.25,70,0.05,891.0
2021-08-09,0.1,0.0,0.25,68,0.05,884.0
2021-08-10,0.1,0.0,0.25,69,0.05,865.0
2021-08-11,0.1,0.0,0.25,68,0.05,904.0
2021-08-12,0.1,0.0,0.25,68,0.05,909.0
2021-08-13,0.1,0.0,0.25,65,0.05,893.0
2021-08-16,0.1,0.0,0.25,66,0.05,906.0
2021-08-17,0.1,0.0,0.25,64,0.05,905.0
2021-08-18,0.09,0.0,0.25,65,0.05,935.0
2021-08-19,0.09,0.0,0.25,67,0.05,925.0
2021-08-20,0.09,0.0,0.25,68,0.05,902.0
2021-08-23,0.09,0.0,0.25,67,0.05,899.0
2021-08-24,0.09,0.0,0.25,66,0.05,923.0
2021-08-25,0.09,0.0,0.25,70,0.05,912.0
2021-08-26,0.09,0.0,0.25,69,0.05,864.0
2021-08-27,0.08,0.0,0.25,71,0.05,920.0
2021-08-30,0.08,0.0,0.25,71,0.05,885.0
2021-08-31,0.06,0.0,0.25,59,0.05,953.0
2021-09-01,0.08,0.0,0.25,72,0.05,936.0
2021-09-02,0.08,0.0,0.25,69,0.05,952.0
2021-09-03,0.08,0.0,0.25,68,0.05,883.0
2021-09-07,0.08,0.0,0.25,71,0.05,945.0
2021-09-08,0.08,0.0,0.25,70,0.05,896.0
2021-09-09,0.08,0.0,0.25,70,0.05,917.0
2021-09-10,0.08,0.0,0.25,67,0.05,891.0
2021-09-13,0.08,0.0,0.25,67,0.05,914.0
2021-09-14,0.08,0.0,0.25,69,0.05,893.0
2021-09-15,0.08,0.0,0.25,69,0.05,942.0
2021-09-16,0.08,0.0,0.25,66,0.05,905.0
2021-09-17,0.08,0.0,0.25,64,0.05,884.0
2021-09-20,0.08,0.0,0.25,69,0.05,928.0
2021-09-21,0.08,0.0,0.25,70,0.05,894.0
2021-09-22,0.08,0.0,0.25,71,0.05,879.0
2021-09-23,0.08,0.0,0.25,70,0.05,878.0
2021-09-24,0.08,0.0,0.25,69,0.05,896.0
2021-09-27,0.08,0.0,0.25,72,0.05,927.0
2021-09-28,0.08,0.0,0.25,71,0.05,883.0
2021-09-29,0.08,0.0,0.25,73,0.05,872.0
2021-09-30,0.06,0.0,0.25,52,0.05,939.0
2021-10-01,0.08,0.0,0.25,68,0.05,928.0
2021-10-04,0.08,0.0,0.25,73,0.05,925.0
2021-10-05,0.08,0.0,0.25,74,0.05,916.0
2021-10-06,0.08,0.0,0.25,75,0.05,902.0
2021-10-07,0.08,0.0,0.25,77,0.05,911.0
2021-10-08,0.08,0.0,0.25,77,0.05,887.0
2021-10-12,0.08,0.0,0.25,76,0.05,925.0
2021-10-13,0.08,0.0,0.25,73,0.05,907.0
2021-10-14,0.08,0.0,0.25,72,0.05,873.0
2021-10-15,0.08,0.0,0.25,73,0.05,889.0
2021-10-18,0.08,0.0,0.25,71,0.05,879.0
2021-10-19,0.08,0.0,0.25,70,0.03,875.0
2021-10-20,0.08,0.0,0.25,72,0.03,857.0
2021-10-21,0.08,0.0,0.25,75,0.03,879.0
2021-10-22,0.08,0.0,0.25,74,0.05,863.0
2021-10-25,0.08,0.0,0.25,78,0.04,858.0
2021-10-26,0.08,0.0,0.25,77,0.05,877.0
2021-10-27,0.08,0.0,0.25,78,0.05,865.0
2021-10-28,0.08,0.0,0.25,84,0.05,871.0
2021-10-29,0.07,0.0,0.25,70,0.05,847.0
2021-11-01,0.08,0.0,0.25,78,0.05,959.0
2021-11-02,0.08,0.0,0.25,78,0.05,910.0
2021-11-03,0.08,0.0,0.25,75,0.05,877.0
2021-11-04,0.08,0.0,0.25,76,0.05,925.0
2021-11-05,0.08,0.0,0.25,79,0.05,875.0
2021-11-08,0.08,0.0,0.25,76,0.05,907.0
2021-11-09,0.08,0.0,0.25,75,0.05,908.0
2021-11-10,0.08,0.0,0.25,72,0.05,871.0
2021-11-12,0.08,0.0,0.25,74,0.05,868.0
2021-11-15,0.08,0.0,0.25,77,0.05,958.0
2021-11-16,0.08,0.0,0.25,76,0.05,924.0
2021-11-17,0.08,0.0,0.25,74,0.05,893.0
2021-11-18,0.08,0.0,0.25,77,0.05,898.0
2021-11-19,0.08,0.0,0.25,74,0.05,895.0
2021-11-22,0.08,0.0,0.25,74,0.05,930.0
2021-11-23,0.08,0.0,0.25,74,0.05,920.0
2021-11-24,0.08,0.0,0.25,79,0.05,887.0
2021-11-26,0.08,0.0,0.25,85,0.05,895.0
2021-11-29,0.08,0.0,0.25,85,0.05,953.0
2021-11-30,0.07,0.0,0.25,77,0.05,1067.0
2021-12-01,0.08,0.0,0.25,76,0.05,1088.0
2021-12-02,0.08,0.0,0.25,78,0.05,1031.0
2021-12-03,0.08,0.0,0.25,77,0.05,1015.0
2021-12-06,0.08,0.0,0.25,74,0.05,994.0
2021-12-07,0.08,0.0,0.25,74,0.05,989.0
2021-12-08,0.08,0.0,0.25,76,0.05,978.0
2021-12-09,0.08,0.0,0.25,76,0.05,973.0
2021-12-10,0.08,0.0,0.25,74,0.05,919.0
2021-12-13,0.08,0.0,0.25,69,0.05,936.0
2021-12-14,0.08,0.0,0.25,73,0.05,919.0
2021-12-15,0.08,0.0,0.25,74,0.05,959.0
2021-12-16,0.08,0.0,0.25,73,0.05,914.0
2021-12-17,0.08,0.0,0.25,71,0.05,910.0
2021-12-20,0.08,0.0,0.25,72,0.04,905.0
2021-12-21,0.08,0.0,0.25,69,0.05,896.0
2021-12-22,0.08,0.0,0.25,69,0.05,893.0
2021-12-23,0.08,0.0,0.25,71,0.05,880.0
2021-12-24,0.08,0.0,0.25,69,,
2021-12-27,0.08,0.0,0.25,75,0.05,888.0
2021-12-28,0.08,0.0,0.25,72,0.05,844.0
2021-12-29,0.08,0.0,0.25,71,0.05,869.0
2021-12-30,0.08,0.0,0.25,77,0.05,836.0
2021-12-31,0.07,0.0,0.25,49,0.05,923.0
2022-01-03,0.08,0.0,0.25,74,0.05,972.0
2022-01-04,0.08,0.0,0.25,74,0.05,976.0
2022-01-05,0.08,0.0,0.25,74,0.05,946.0
2022-01-06,0.08,0.0,0.25,73,0.05,942.0
2022-01-07,0.08,0.0,0.25,71,0.05,929.0
2022-01-10,0.08,0.0,0.25,71,0.05,919.0
2022-01-11,0.08,0.0,0.25,71,0.05,935.0
2022-01-12,0.08,0.0,0.25,70,0.05,891.0
2022-01-13,0.08,0.0,0.25,71,0.05,892.0
2022-01-14,0.08,0.0,0.25,72,0.05,874.0
2022-01-18,0.08,0.0,0.25,71,0.05,954.0
2022-01-19,0.08,0.0,0.25,73,0.05,917.0
2022-01-20,0.08,0.0,0.25,75,0.04,927.0
2022-01-21,0.08,0.0,0.25,76,0.05,875.0
2022-01-24,0.08,0.0,0.25,75,0.04,886.0
2022-01-25,0.08,0.0,0.25,78,0.05,943.0
2022-01-26,0.08,0.0,0.25,78,0.04,926.0
2022-01-27,0.08,0.0,0.25,80,0.04,913.0
2022-01-28,0.08,0.0,0.25,82,0.05,908.0
2022-01-31,0.08,0.0,0.25,78,0.05,1019.0
2022-02-01,0.08,0.0,0.25,75,0.05,985.0
2022-02-02,0.08,0.0,0.25,72,0.05,927.0
2022-02-03,0.08,0.0,0.25,72,0.05,935.0
2022-02-04,0.08,0.0,0.25,69,0.05,891.0
2022-02-07,0.08,0.0,0.25,72,0.05,905.0
2022-02-08,0.08,0.0,0.25,70,0.05,896.0
2022-02-09,0.08,0.0,0.25,70,0.04,911.0
2022-02-10,0.08,0.0,0.25,71,0.05,914.0
2022-02-11,0.08,0.0,0.25,67,0.05,903.0
2022-02-14,0.08,0.0,0.25,63,0.05,895.0
2022-02-15,0.08,0.0,0.25,65,0.05,972.0
2022-02-16,0.08,0.0,0.25,70,0.05,960.0
2022-02-17,0.08,0.0,0.25,73,0.05,911.0
2022-02-18,0.08,0.0,0.25,71,0.05,931.0
2022-02-22,0.08,0.0,0.25,72,0.05,930.0
2022-02-23,0.08,0.0,0.25,72,0.05,902.0
2022-02-24,0.08,0.0,0.25,64,0.05,924.0
2022-02-25,0.08,0.0,0.25,69,0.05,962.0
2022-02-28,0.08,0.0,0.25,65,0.05,1052.0
2022-03-01,0.08,0.0,0.25,64,0.05,1062.0
2022-03-02,0.08,0.0,0.25,70,0.05,1023.0
2022-03-03,0.08,0.0,0.25,68,0.05,1001.0
2022-03-04,0.08,0.0,0.25,72,0.05,961.0
2022-03-07,0.08,0.0,0.25,71,0.05,990.0
2022-03-08,0.08,0.0,0.25,69,0.05,948.0
2022-03-09,0.08,0.0,0.25,71,0.05,952.0
2022-03-10,0.08,0.0,0.25,72,0.05,977.0
2022-03-11,0.08,0.0,0.25,73,0.05,980.0
2022-03-14,0.08,0.0,0.25,72,0.05,980.0
2022-03-15,0.08,0.0,0.25,76,0.05,1014.0
2022-03-16,0.08,0.0,0.25,74,0.05,1016.0
2022-03-17,0.33,0.25,0.5,73,0.3,1003.0
2022-03-18,0.33,0.25,0.5,72,0.3,968.0
2022-03-21,0.33,0.25,0.5,76,0.29,989.0
2022-03-22,0.33,0.25,0.5,76,0.28,931.0
2022-03-23,0.33,0.25,0.5,74,0.27,908.0
2022-03-24,0.33,0.25,0.5,74,0.27,900.0
2022-03-25,0.33,0.25,0.5,78,0.28,901.0
2022-03-28,0.33,0.25,0.5,77,0.28,899.0
2022-03-29,0.33,0.25,0.5,83,0.28,874.0
2022-03-30,0.33,0.25,0.5,80,0.27,900.0
2022-03-31,0.33,0.25,0.5,57,0.29,979.0
2022-04-01,0.33,0.25,0.5,78,0.3,974.0
2022-04-04,0.33,0.25,0.5,78,0.3,940.0
2022-04-05,0.33,0.25,0.5,79,0.3,929.0
2022-04-06,0.33,0.25,0.5,77,0.3,905.0
2022-04-07,0.33,0.25,0.5,78,0.3,906.0
2022-04-08,0.33,0.25,0.5,77,0.3,905.0
2022-04-11,0.33,0.25,0.5,80,0.3,886.0
2022-04-12,0.33,0.25,0.5,80,0.29,883.0
2022-04-13,0.33,0.25,0.5,77,0.29,874.0
2022-04-14,0.33,0.25,0.5,71,0.29,886.0
2022-04-15,0.33,0.25,0.5,75,,
2022-04-18,0.33,0.25,0.5,76,0.29,896.0
2022-04-19,0.33,0.25,0.5,70,0.28,895.0
2022-04-20,0.33,0.25,0.5,71,0.27,912.0
2022-04-21,0.33,0.25,0.5,70,0.26,906.0
2022-04-22,0.33,0.25,0.5,74,0.27,901.0
2022-04-25,0.33,0.25,0.5,78,0.27,898.0
2022-04-26,0.33,0.25,0.5,73,0.27,897.0
2022-04-27,0.33,0.25,0.5,78,0.28,882.0
2022-04-28,0.33,0.25,0.5,81,0.28,892.0
2022-04-29,0.33,0.25,0.5,76,0.28,915.0
2022-05-02,0.33,0.25,0.5,77,0.3,984.0
2022-05-03,0.33,0.25,0.5,79,0.3,923.0
2022-05-04,0.33,0.25,0.5,77,0.3,947.0
2022-05-05,0.83,0.75,1.0,74,0.79,992.0
2022-05-06,0.83,0.75,1.0,81,0.78,979.0
2022-05-09,0.83,0.75,1.0,75,0.78,931.0
2022-05-10,0.83,0.75,1.0,72,0.78,914.0
2022-05-11,0.83,0.75,1.0,78,0.78,925.0
2022-05-12,0.83,0.75,1.0,81,0.79,922.0
2022-05-13,0.83,0.75,1.0,84,0.79,917.0
2022-05-16,0.83,0.75,1.0,84,0.8,1013.0
2022-05-17,0.83,0.75,1.0,80,0.8,977.0
2022-05-18,0.83,0.75,1.0,75,0.79,939.0
2022-05-19,0.83,0.75,1.0,76,0.79,982.0
2022-05-20,0.83,0.75,1.0,82,0.78,943.0
2022-05-23,0.83,0.75,1.0,82,0.78,984.0
2022-05-24,0.83,0.75,1.0,82,0.78,968.0
2022-05-25,0.83,0.75,1.0,85,0.78,942.0
2022-05-26,0.83,0.75,1.0,81,0.78,955.0
2022-05-27,0.83,0.75,1.0,79,0.78,963.0
2022-05-31,0.83,0.75,1.0,76,0.79,1040.0
2022-06-01,0.83,0.75,1.0,85,0.8,994.0
2022-06-02,0.83,0.75,1.0,83,0.79,974.0
2022-06-03,0.83,0.75,1.0,85,0.78,954.0
2022-06-06,0.83,0.75,1.0,85,0.78,959.0
2022-06-07,0.83,0.75,1.0,82,0.77,947.0
2022-06-08,0.83,0.75,1.0,79,0.76,959.0
2022-06-09,0.83,0.75,1.0,79,0.75,985.0
2022-06-10,0.83,0.75,1.0,76,0.75,960.0
2022-06-13,0.83,0.75,1.0,75,0.73,970.0
2022-06-14,0.83,0.75,1.0,81,0.69,925.0
2022-06-15,0.83,0.75,1.0,83,0.7,917.0
2022-06-16,1.58,1.5,1.75,80,1.45,976.0
2022-06-17,1.58,1.5,1.75,84,1.45,923.0
2022-06-21,1.58,1.5,1.75,67,1.45,957.0
2022-06-22,1.58,1.5,1.75,85,1.45,942.0
2022-06-23,1.58,1.5,1.75,84,1.44,944.0
2022-06-24,1.58,1.5,1.75,91,1.46,926.0
2022-06-27,1.58,1.5,1.75,90,1.5,945.0
2022-06-28,1.58,1.5,1.75,95,1.52,929.0
2022-06-29,1.58,1.5,1.75,85,1.51,940.0
2022-06-30,1.58,1.5,1.75,67,1.5,995.0
2022-07-01,1.58,1.5,1.75,90,1.52,1001.0
2022-07-05,1.58,1.5,1.75,94,1.54,1042.0
2022-07-06,1.58,1.5,1.75,95,1.54,966.0
2022-07-07,1.58,1.5,1.75,92,1.54,966.0
2022-07-08,1.58,1.5,1.75,99,1.53,942.0
2022-07-11,1.58,1.5,1.75,93,1.53,930.0
2022-07-12,1.58,1.5,1.75,98,1.54,941.0
2022-07-13,1.58,1.5,1.75,96,1.53,924.0
2022-07-14,1.58,1.5,1.75,91,1.53,934.0
2022-07-15,1.58,1.5,1.75,97,1.54,950.0
2022-07-18,1.58,1.5,1.75,97,1.54,947.0
2022-07-19,1.58,1.5,1.75,90,1.54,933.0
2022-07-20,1.58,1.5,1.75,95,1.53,916.0
2022-07-21,1.58,1.5,1.75,96,1.53,960.0
2022-07-22,1.58,1.5,1.75,96,1.52,931.0
2022-07-25,1.58,1.5,1.75,94,1.53,985.0
2022-07-26,1.58,1.5,1.75,97,1.53,951.0
2022-07-27,1.58,1.5,1.75,98,1.53,941.0
2022-07-28,2.33,2.25,2.5,90,2.28,965.0
2022-07-29,2.32,2.25,2.5,86,2.27,949.0
2022-08-01,2.33,2.25,2.5,93,2.28,1035.0
2022-08-02,2.33,2.25,2.5,95,2.3,1015.0
2022-08-03,2.33,2.25,2.5,92,2.29,984.0
2022-08-04,2.33,2.25,2.5,91,2.29,981.0
2022-08-05,2.33,2.25,2.5,91,2.28,965.0
2022-08-08,2.33,2.25,2.5,89,2.28,950.0
2022-08-09,2.33,2.25,2.5,90,2.29,947.0
2022-08-10,2.33,2.25,2.5,93,2.28,970.0
2022-08-11,2.33,2.25,2.5,94,2.28,965.0
2022-08-12,2.33,2.25,2.5,97,2.28,966.0
2022-08-15,2.33,2.25,2.5,97,2.29,1014.0
2022-08-16,2.33,2.25,2.5,94,2.29,974.0
2022-08-17,2.33,2.25,2.5,92,2.29,966.0
2022-08-18,2.33,2.25,2.5,91,2.28,973.0
2022-08-19,2.33,2.25,2.5,94,2.28,960.0
2022-08-22,2.33,2.25,2.5,91,2.28,994.0
2022-08-23,2.33,2.25,2.5,91,2.27,952.0
2022-08-24,2.33,2.25,2.5,94,2.27,969.0
2022-08-25,2.33,2.25,2.5,96,2.28,970.0
2022-08-26,2.33,2.25,2.5,98,2.28,953.0
2022-08-29,2.33,2.25,2.5,89,2.28,963.0
2022-08-30,2.33,2.25,2.5,90,2.29,948.0
2022-08-31,2.33,2.25,2.5,86,2.29,1024.0
2022-09-01,2.33,2.25,2.5,100,2.29,979.0
2022-09-02,2.33,2.25,2.5,100,2.29,965.0
2022-09-06,2.33,2.25,2.5,94,2.29,989.0
2022-09-07,2.33,2.25,2.5,97,2.28,958.0
2022-09-08,2.33,2.25,2.5,94,2.28,981.0
2022-09-09,2.33,2.25,2.5,101,2.28,958.0
2022-09-12,2.33,2.25,2.5,93,2.28,980.0
2022-09-13,2.33,2.25,2.5,94,2.28,957.0
2022-09-14,2.33,2.25,2.5,95,2.27,951.0
2022-09-15,2.33,2.25,2.5,96,2.28,981.0
2022-09-16,2.33,2.25,2.5,96,2.27,952.0
2022-09-19,2.33,2.25,2.5,93,2.27,948.0
2022-09-20,2.33,2.25,2.5,91,2.26,967.0
2022-09-21,2.33,2.25,2.5,99,2.25,951.0
2022-09-22,3.08,3.0,3.25,99,2.99,1023.0
2022-09-23,3.08,3.0,3.25,109,2.99,968.0
2022-09-26,3.08,3.0,3.25,111,2.99,976.0
2022-09-27,3.08,3.0,3.25,111,2.98,949.0
2022-09-28,3.08,3.0,3.25,110,2.98,911.0
2022-09-29,3.08,3.0,3.25,103,2.96,906.0
2022-09-30,3.08,3.0,3.25,90,2.98,956.0
2022-10-03,3.08,3.0,3.25,115,3.0,1009.0
2022-10-04,3.08,3.0,3.25,107,3.04,996.0
2022-10-05,3.08,3.0,3.25,104,3.04,992.0
2022-10-06,3.08,3.0,3.25,109,3.05,985.0
2022-10-07,3.08,3.0,3.25,113,3.05,968.0
2022-10-11,3.08,3.0,3.25,104,3.05,993.0
2022-10-12,3.08,3.0,3.25,110,3.04,972.0
2022-10-13,3.08,3.0,3.25,106,3.04,1000.0
2022-10-14,3.08,3.0,3.25,110,3.04,958.0
2022-10-17,3.08,3.0,3.25,103,3.05,979.0
2022-10-18,3.08,3.0,3.25,103,3.04,967.0
2022-10-19,3.08,3.0,3.25,104,3.04,919.0
2022-10-20,3.08,3.0,3.25,92,3.03,963.0
2022-10-21,3.08,3.0,3.25,99,3.02,955.0
2022-10-24,3.08,3.0,3.25,99,3.01,962.0
2022-10-25,3.08,3.0,3.25,103,3.02,961.0
2022-10-26,3.08,3.0,3.25,99,3.03,972.0
2022-10-27,3.08,3.0,3.25,103,3.04,973.0
2022-10-28,3.08,3.0,3.25,107,3.05,951.0
2022-10-31,3.08,3.0,3.25,82,3.05,1075.0
2022-11-01,3.08,3.0,3.25,108,3.05,1079.0
2022-11-02,3.08,3.0,3.25,103,3.05,1021.0
2022-11-03,3.83,3.75,4.0,97,3.8,1074.0
2022-11-04,3.83,3.75,4.0,105,3.8,1014.0
2022-11-07,3.83,3.75,4.0,99,3.78,981.0
2022-11-08,3.83,3.75,4.0,101,3.78,978.0
2022-11-09,3.83,3.75,4.0,100,3.78,969.0
2022-11-10,3.83,3.75,4.0,97,3.78,987.0
2022-11-14,3.83,3.75,4.0,99,3.79,1041.0
2022-11-15,3.83,3.75,4.0,97,3.8,1149.0
2022-11-16,3.83,3.75,4.0,98,3.81,1096.0
2022-11-17,3.83,3.75,4.0,99,3.8,1075.0
2022-11-18,3.83,3.75,4.0,102,3.8,1037.0
2022-11-21,3.83,3.75,4.0,94,3.8,1015.0
2022-11-22,3.83,3.75,4.0,99,3.8,1038.0
2022-11-23,3.83,3.75,4.0,108,3.79,984.0
2022-11-25,3.83,3.75,4.0,105,3.8,1003.0
2022-11-28,3.83,3.75,4.0,97,3.8,1058.0
2022-11-29,3.83,3.75,4.0,95,3.81,1026.0
2022-11-30,3.83,3.75,4.0,92,3.82,1113.0
2022-12-01,3.83,3.75,4.0,99,3.82,1174.0
2022-12-02,3.83,3.75,4.0,108,3.81,1132.0
2022-12-05,3.83,3.75,4.0,107,3.81,1096.0
2022-12-06,3.83,3.75,4.0,110,3.8,1088.0
2022-12-07,3.83,3.75,4.0,103,3.8,1088.0
2022-12-08,3.83,3.75,4.0,99,3.8,1087.0
2022-12-09,3.83,3.75,4.0,102,3.8,1058.0
2022-12-12,3.83,3.75,4.0,97,3.8,1042.0
2022-12-13,3.83,3.75,4.0,100,3.8,1051.0
2022-12-14,3.83,3.75,4.0,101,3.8,1045.0
2022-12-15,4.33,4.25,4.5,104,4.32,1144.0
2022-12-16,4.33,4.25,4.5,99,4.32,1091.0
2022-12-19,4.33,4.25,4.5,91,4.3,1087.0
2022-12-20,4.33,4.25,4.5,99,4.3,1048.0
2022-12-21,4.33,4.25,4.5,98,4.3,1031.0
2022-12-22,4.33,4.25,4.5,99,4.3,1013.0
2022-12-23,4.33,4.25,4.5,107,4.3,991.0
2022-12-27,4.33,4.25,4.5,109,4.3,1048.0
2022-12-28,4.33,4.25,4.5,99,4.3,1005.0
2022-12-29,4.33,4.25,4.5,94,4.3,1037.0
2022-12-30,4.33,4.25,4.5,67,4.3,1004.0
2023-01-03,4.33,4.25,4.5,105,4.31,1257.0
2023-01-04,4.33,4.25,4.5,108,4.3,1136.0
2023-01-05,4.33,4.25,4.5,107,4.31,1134.0
2023-01-06,4.33,4.25,4.5,114,4.31,1118.0
2023-01-09,4.33,4.25,4.5,106,4.31,1127.0
2023-01-10,4.33,4.25,4.5,109,4.31,1124.0
2023-01-11,4.33,4.25,4.5,110,4.3,1142.0
2023-01-12,4.33,4.25,4.5,107,4.3,1108.0
2023-01-13,4.33,4.25,4.5,110,4.3,1101.0
2023-01-17,4.33,4.25,4.5,94,4.31,1248.0
2023-01-18,4.33,4.25,4.5,106,4.3,1193.0
2023-01-19,4.33,4.25,4.5,100,4.31,1231.0
2023-01-20,4.33,4.25,4.5,99,4.3,1169.0
2023-01-23,4.33,4.25,4.5,96,4.3,1150.0
2023-01-24,4.33,4.25,4.5,104,4.3,1163.0
2023-01-25,4.33,4.25,4.5,111,4.31,1157.0
2023-01-26,4.33,4.25,4.5,113,4.3,1165.0
2023-01-27,4.33,4.25,4.5,120,4.3,1165.0
2023-01-30,4.33,4.25,4.5,109,4.3,1158.0
2023-01-31,4.33,4.25,4.5,98,4.31,1281.0
2023-02-01,4.33,4.25,4.5,106,4.31,1199.0
2023-02-02,4.58,4.5,4.75,99,4.56,1290.0
2023-02-03,4.58,4.5,4.75,104,4.55,1216.0
2023-02-06,4.58,4.5,4.75,99,4.55,1209.0
2023-02-07,4.58,4.5,4.75,103,4.55,1208.0
2023-02-08,4.58,4.5,4.75,106,4.55,1184.0
2023-02-09,4.57,4.5,4.75,111,4.55,1212.0
2023-02-10,4.58,4.5,4.75,112,4.55,1186.0
2023-02-13,4.58,4.5,4.75,107,4.55,1153.0
2023-02-14,4.58,4.5,4.75,105,4.55,1216.0
2023-02-15,4.58,4.5,4.75,111,4.55,1239.0
2023-02-16,4.58,4.5,4.75,105,4.55,1244.0
2023-02-17,4.58,4.5,4.75,107,4.55,1209.0
2023-02-21,4.58,4.5,4.75,100,4.55,1164.0
2023-02-22,4.58,4.5,4.75,109,4.55,1129.0
2023-02-23,4.58,4.5,4.75,106,4.55,1137.0
2023-02-24,4.58,4.5,4.75,109,4.55,1122.0
2023-02-27,4.57,4.5,4.75,108,4.55,1145.0
2023-02-28,4.57,4.5,4.75,103,4.55,1230.0
2023-03-01,4.58,4.5,4.75,117,4.55,1172.0
2023-03-02,4.57,4.5,4.75,108,4.55,1150.0
2023-03-03,4.57,4.5,4.75,114,4.55,1120.0
2023-03-06,4.57,4.5,4.75,114,4.55,1129.0
2023-03-07,4.57,4.5,4.75,118,4.55,1151.0
2023-03-08,4.57,4.5,4.75,107,4.55,1096.0
2023-03-09,4.57,4.5,4.75,115,4.55,1120.0
2023-03-10,4.57,4.5,4.75,81,4.55,1088.0
2023-03-13,4.58,4.5,4.75,41,4.55,1148.0
2023-03-14,4.58,4.5,4.75,79,4.55,1203.0
2023-03-15,4.58,4.5,4.75,89,4.58,1252.0
2023-03-16,4.58,4.5,4.75,83,4.57,1272.0
2023-03-17,4.58,4.5,4.75,85,4.55,1265.0
2023-03-20,4.58,4.5,4.75,92,4.55,1288.0
2023-03-21,4.58,4.5,4.75,92,4.55,1247.0
2023-03-22,4.58,4.5,4.75,100,4.55,1203.0
2023-03-23,4.83,4.75,5.0,92,4.8,1355.0
2023-03-24,4.83,4.75,5.0,92,4.8,1290.0
2023-03-27,4.83,4.75,5.0,92,4.81,1365.0
2023-03-28,4.83,4.75,5.0,98,4.84,1345.0
2023-03-29,4.83,4.75,5.0,97,4.83,1287.0
2023-03-30,4.83,4.75,5.0,95,4.82,1319.0
2023-03-31,4.83,4.75,5.0,86,4.87,1408.0
2023-04-03,4.83,4.75,5.0,105,4.84,1477.0
2023-04-04,4.83,4.75,5.0,103,4.83,1427.0
2023-04-05,4.83,4.75,5.0,102,4.81,1427.0
2023-04-06,4.83,4.75,5.0,98,4.81,1431.0
2023-04-07,4.83,4.75,5.0,100,,
2023-04-10,4.83,4.75,5.0,106,4.81,1366.0
2023-04-11,4.83,4.75,5.0,108,4.8,1365.0
2023-04-12,4.83,4.75,5.0,114,4.8,1358.0
2023-04-13,4.83,4.75,5.0,109,4.8,1378.0
2023-04-14,4.83,4.75,5.0,113,4.8,1312.0
2023-04-17,4.83,4.75,5.0,106,4.8,1406.0
2023-04-18,4.83,4.75,5.0,112,4.8,1364.0
2023-04-19,4.83,4.75,5.0,109,4.8,1307.0
2023-04-20,4.83,4.75,5.0,115,4.8,1281.0
2023-04-21,4.83,4.75,5.0,114,4.8,1324.0
2023-04-24,4.83,4.75,5.0,110,4.8,1298.0
2023-04-25,4.83,4.75,5.0,112,4.8,1329.0
2023-04-26,4.83,4.75,5.0,115,4.8,1345.0
2023-04-27,4.83,4.75,5.0,119,4.81,1337.0
2023-04-28,4.83,4.75,5.0,113,4.81,1378.0
2023-05-01,4.83,4.75,5.0,117,4.81,1577.0
2023-05-02,4.83,4.75,5.0,118,4.81,1537.0
2023-05-03,4.83,4.75,5.0,115,4.81,1544.0
2023-05-04,5.08,5.0,5.25,114,5.06,1590.0
2023-05-05,5.08,5.0,5.25,112,5.06,1517.0
2023-05-08,5.08,5.0,5.25,111,5.06,1497.0
2023-05-09,5.08,5.0,5.25,112,5.06,1482.0
2023-05-10,5.08,5.0,5.25,120,5.06,1410.0
2023-05-11,5.08,5.0,5.25,117,5.05,1372.0
2023-05-12,5.08,5.0,5.25,124,5.05,1352.0
2023-05-15,5.08,5.0,5.25,126,5.06,1497.0
2023-05-16,5.08,5.0,5.25,124,5.05,1466.0
2023-05-17,5.08,5.0,5.25,130,5.05,1401.0
2023-05-18,5.08,5.0,5.25,127,5.05,1401.0
2023-05-19,5.08,5.0,5.25,132,5.05,1363.0
2023-05-22,5.08,5.0,5.25,124,5.05,1434.0
2023-05-23,5.08,5.0,5.25,126,5.05,1383.0
2023-05-24,5.08,5.0,5.25,126,5.05,1401.0
2023-05-25,5.08,5.0,5.25,131,5.06,1537.0
2023-05-26,5.08,5.0,5.25,136,5.06,1476.0
2023-05-30,5.08,5.0,5.25,127,5.06,1530.0
2023-05-31,5.08,5.0,5.25,118,5.08,1623.0
2023-06-01,5.08,5.0,5.25,131,5.08,1627.0
2023-06-02,5.08,5.0,5.25,141,5.07,1475.0
2023-06-05,5.08,5.0,5.25,136,5.06,1472.0
2023-06-06,5.08,5.0,5.25,144,5.05,1414.0
2023-06-07,5.08,5.0,5.25,137,5.05,1429.0
2023-06-08,5.08,5.0,5.25,136,5.05,1402.0
2023-06-09,5.08,5.0,5.25,133,5.05,1392.0
2023-06-12,5.08,5.0,5.25,134,5.05,1399.0
2023-06-13,5.08,5.0,5.25,131,5.05,1375.0
2023-06-14,5.08,5.0,5.25,128,5.05,1397.0
2023-06-15,5.07,5.0,5.25,128,5.06,1527.0
2023-06-16,5.08,5.0,5.25,129,5.05,1405.0
2023-06-20,5.07,5.0,5.25,134,5.05,1427.0
2023-06-21,5.07,5.0,5.25,130,5.05,1387.0
2023-06-22,5.07,5.0,5.25,132,5.05,1354.0
2023-06-23,5.07,5.0,5.25,135,5.05,1389.0
2023-06-26,5.07,5.0,5.25,138,5.05,1429.0
2023-06-27,5.07,5.0,5.25,132,5.05,1481.0
2023-06-28,5.07,5.0,5.25,140,5.06,1499.0
2023-06-29,5.07,5.0,5.25,135,5.06,1542.0
2023-06-30,5.08,5.0,5.25,97,5.09,1549.0
2023-07-03,5.08,5.0,5.25,124,5.06,1579.0
2023-07-05,5.08,5.0,5.25,123,5.06,1589.0
2023-07-06,5.08,5.0,5.25,119,5.06,1573.0
2023-07-07,5.08,5.0,5.25,127,5.06,1547.0
2023-07-10,5.07,5.0,5.25,125,5.06,1506.0
2023-07-11,5.08,5.0,5.25,121,5.06,1439.0
2023-07-12,5.08,5.0,5.25,116,5.05,1361.0
2023-07-13,5.08,5.0,5.25,122,5.06,1445.0
2023-07-14,5.08,5.0,5.25,115,5.05,1448.0
2023-07-17,5.08,5.0,5.25,112,5.06,1524.0
2023-07-18,5.08,5.0,5.25,106,5.06,1482.0
2023-07-19,5.08,5.0,5.25,106,5.05,1456.0
2023-07-20,5.08,5.0,5.25,111,5.06,1460.0
2023-07-21,5.08,5.0,5.25,110,5.05,1400.0
2023-07-24,5.08,5.0,5.25,101,5.05,1365.0
2023-07-25,5.08,5.0,5.25,89,5.06,1470.0
2023-07-26,5.08,5.0,5.25,98,5.06,1452.0
2023-07-27,5.33,5.25,5.5,105,5.31,1501.0
2023-07-28,5.33,5.25,5.5,106,5.3,1419.0
2023-07-31,5.33,5.25,5.5,103,5.31,1551.0
2023-08-01,5.33,5.25,5.5,108,5.31,1495.0
2023-08-02,5.33,5.25,5.5,105,5.3,1416.0
2023-08-03,5.33,5.25,5.5,105,5.3,1413.0
2023-08-04,5.33,5.25,5.5,112,5.3,1369.0
2023-08-07,5.33,5.25,5.5,106,5.3,1349.0
2023-08-08,5.33,5.25,5.5,110,5.3,1340.0
2023-08-09,5.33,5.25,5.5,111,5.3,1322.0
2023-08-10,5.33,5.25,5.5,108,5.3,1360.0
2023-08-11,5.33,5.25,5.5,105,5.3,1347.0
2023-08-14,5.33,5.25,5.5,103,5.3,1333.0
2023-08-15,5.33,5.25,5.5,102,5.3,1413.0
2023-08-16,5.33,5.25,5.5,96,5.3,1391.0
2023-08-17,5.33,5.25,5.5,98,5.3,1343.0
2023-08-18,5.33,5.25,5.5,100,5.3,1314.0
2023-08-21,5.33,5.25,5.5,100,5.3,1322.0
2023-08-22,5.33,5.25,5.5,101,5.3,1307.0
2023-08-23,5.33,5.25,5.5,105,5.3,1270.0
2023-08-24,5.33,5.25,5.5,105,5.3,1344.0
2023-08-25,5.33,5.25,5.5,103,5.3,1344.0
2023-08-28,5.33,5.25,5.5,107,5.3,1384.0
2023-08-29,5.33,5.25,5.5,102,5.3,1314.0
2023-08-30,5.33,5.25,5.5,100,5.3,1311.0
2023-08-31,5.33,5.25,5.5,97,5.31,1549.0
2023-09-01,5.33,5.25,5.5,104,5.31,1458.0
2023-09-05,5.33,5.25,5.5,102,5.31,1472.0
2023-09-06,5.33,5.25,5.5,100,5.3,1391.0
2023-09-07,5.33,5.25,5.5,105,5.31,1479.0
2023-09-08,5.33,5.25,5.5,104,5.3,1420.0
2023-09-11,5.33,5.25,5.5,105,5.3,1378.0
2023-09-12,5.33,5.25,5.5,108,5.3,1448.0
2023-09-13,5.33,5.25,5.5,104,5.3,1385.0
2023-09-14,5.33,5.25,5.5,92,5.3,1436.0
2023-09-15,5.33,5.25,5.5,96,5.31,1586.0
2023-09-18,5.33,5.25,5.5,94,5.31,1565.0
2023-09-19,5.33,5.25,5.5,90,5.31,1562.0
2023-09-20,5.33,5.25,5.5,89,5.3,1492.0
2023-09-21,5.33,5.25,5.5,87,5.3,1522.0
2023-09-22,5.33,5.25,5.5,94,5.3,1511.0
2023-09-25,5.33,5.25,5.5,93,5.31,1524.0
2023-09-26,5.33,5.25,5.5,96,5.31,1561.0
2023-09-27,5.33,5.25,5.5,99,5.32,1388.0
2023-09-28,5.33,5.25,5.5,104,5.31,1535.0
2023-09-29,5.33,5.25,5.5,86,5.31,1537.0
2023-10-02,5.33,5.25,5.5,89,5.32,1513.0
2023-10-03,5.33,5.25,5.5,94,5.33,1498.0
2023-10-04,5.33,5.25,5.5,103,5.32,1436.0
2023-10-05,5.33,5.25,5.5,102,5.32,1520.0
2023-10-06,5.33,5.25,5.5,100,5.31,1404.0
2023-10-10,5.33,5.25,5.5,95,5.31,1522.0
2023-10-11,5.33,5.25,5.5,94,5.31,1465.0
2023-10-12,5.33,5.25,5.5,103,5.31,1467.0
2023-10-13,5.33,5.25,5.5,99,5.31,1383.0
2023-10-16,5.33,5.25,5.5,96,5.31,1450.0
2023-10-17,5.33,5.25,5.5,93,5.31,1427.0
2023-10-18,5.33,5.25,5.5,89,5.3,1396.0
2023-10-19,5.33,5.25,5.5,90,5.3,1398.0
2023-10-20,5.33,5.25,5.5,87,5.3,1396.0
2023-10-23,5.33,5.25,5.5,88,5.3,1330.0
2023-10-24,5.33,5.25,5.5,83,5.3,1387.0
2023-10-25,5.33,5.25,5.5,93,5.3,1439.0
2023-10-26,5.33,5.25,5.5,100,5.31,1506.0
2023-10-27,5.33,5.25,5.5,94,5.31,1496.0
2023-10-30,5.33,5.25,5.5,97,5.31,1470.0
2023-10-31,5.33,5.25,5.5,86,5.35,1582.0
2023-11-01,5.33,5.25,5.5,92,5.32,1621.0
2023-11-02,5.33,5.25,5.5,99,5.33,1640.0
2023-11-03,5.33,5.25,5.5,102,5.32,1579.0
2023-11-06,5.33,5.25,5.5,101,5.32,1559.0
2023-11-07,5.33,5.25,5.5,104,5.32,1546.0
2023-11-08,5.33,5.25,5.5,101,5.32,1470.0
2023-11-09,5.33,5.25,5.5,105,5.32,1526.0
2023-11-10,5.33,5.25,5.5,106,5.32,1469.0
2023-11-13,5.33,5.25,5.5,102,5.32,1494.0
2023-11-14,5.33,5.25,5.5,102,5.32,1533.0
2023-11-15,5.33,5.25,5.5,96,5.32,1605.0
2023-11-16,5.33,5.25,5.5,99,5.32,1640.0
2023-11-17,5.33,5.25,5.5,98,5.32,1587.0
2023-11-20,5.33,5.25,5.5,101,5.31,1621.0
2023-11-21,5.33,5.25,5.5,102,5.31,1603.0
2023-11-22,5.33,5.25,5.5,102,5.31,1547.0
2023-11-24,5.33,5.25,5.5,101,5.32,1521.0
2023-11-27,5.33,5.25,5.5,97,5.32,1602.0
2023-11-28,5.33,5.25,5.5,97,5.32,1593.0
2023-11-29,5.33,5.25,5.5,96,5.31,1578.0
2023-11-30,5.33,5.25,5.5,91,5.33,1850.0
2023-12-01,5.33,5.25,5.5,107,5.39,1650.0
2023-12-04,5.33,5.25,5.5,103,5.37,1631.0
2023-12-05,5.33,5.25,5.5,103,5.33,1792.0
2023-12-06,5.33,5.25,5.5,103,5.32,1707.0
2023-12-07,5.33,5.25,5.5,102,5.32,1702.0
2023-12-08,5.33,5.25,5.5,101,5.32,1610.0
2023-12-11,5.33,5.25,5.5,102,5.32,1577.0
2023-12-12,5.33,5.25,5.5,106,5.31,1713.0
2023-12-13,5.33,5.25,5.5,106,5.31,1635.0
2023-12-14,5.33,5.25,5.5,108,5.31,1695.0
2023-12-15,5.33,5.25,5.5,101,5.32,1776.0
2023-12-18,5.33,5.25,5.5,95,5.32,1665.0
2023-12-19,5.33,5.25,5.5,96,5.31,1757.0
2023-12-20,5.33,5.25,5.5,97,5.31,1678.0
2023-12-21,5.33,5.25,5.5,101,5.31,1720.0
2023-12-22,5.33,5.25,5.5,90,5.32,1667.0
2023-12-26,5.33,5.25,5.5,96,5.35,1644.0
2023-12-27,5.33,5.25,5.5,88,5.39,1722.0
2023-12-28,5.33,5.25,5.5,81,5.4,1832.0
2023-12-29,5.33,5.25,5.5,59,5.38,1702.0
2024-01-02,5.33,5.25,5.5,83,5.4,1948.0
2024-01-03,5.33,5.25,5.5,87,5.39,1832.0
2024-01-04,5.33,5.25,5.5,86,5.32,1930.0
2024-01-05,5.33,5.25,5.5,84,5.31,1674.0
2024-01-08,5.33,5.25,5.5,86,5.31,1658.0
2024-01-09,5.33,5.25,5.5,91,5.31,1693.0
2024-01-10,5.33,5.25,5.5,91,5.31,1660.0
2024-01-11,5.33,5.25,5.5,96,5.31,1648.0
2024-01-12,5.33,5.25,5.5,93,5.31,1590.0
2024-01-16,5.33,5.25,5.5,85,5.32,1854.0
2024-01-17,5.33,5.25,5.5,88,5.32,1764.0
2024-01-18,5.33,5.25,5.5,87,5.31,1746.0
2024-01-19,5.33,5.25,5.5,94,5.31,1688.0
2024-01-22,5.33,5.25,5.5,92,5.31,1595.0
2024-01-23,5.33,5.25,5.5,97,5.31,1686.0
2024-01-24,5.33,5.25,5.5,96,5.31,1596.0
2024-01-25,5.33,5.25,5.5,101,5.32,1768.0
2024-01-26,5.33,5.25,5.5,100,5.32,1712.0
2024-01-29,5.33,5.25,5.5,92,5.31,1598.0
2024-01-30,5.33,5.25,5.5,92,5.31,1642.0
2024-01-31,5.33,5.25,5.5,75,5.32,1874.0
2024-02-01,5.33,5.25,5.5,81,5.32,1897.0
2024-02-02,5.33,5.25,5.5,91,5.32,1849.0
2024-02-05,5.33,5.25,5.5,96,5.31,1731.0
2024-02-06,5.33,5.25,5.5,97,5.31,1738.0
2024-02-07,5.33,5.25,5.5,96,5.31,1797.0
2024-02-08,5.33,5.25,5.5,96,5.31,1669.0
2024-02-09,5.33,5.25,5.5,96,5.31,1581.0
2024-02-12,5.33,5.25,5.5,93,5.31,1725.0
2024-02-13,5.33,5.25,5.5,98,5.31,1714.0
2024-02-14,5.33,5.25,5.5,107,5.3,1596.0
2024-02-15,5.33,5.25,5.5,112,5.31,1746.0
2024-02-16,5.33,5.25,5.5,110,5.3,1666.0
2024-02-20,5.33,5.25,5.5,104,5.3,1610.0
2024-02-21,5.33,5.25,5.5,101,5.3,1607.0
2024-02-22,5.33,5.25,5.5,103,5.3,1620.0
2024-02-23,5.33,5.25,5.5,103,5.31,1736.0
2024-02-26,5.33,5.25,5.5,98,5.31,1739.0
2024-02-27,5.33,5.25,5.5,96,5.31,1741.0
2024-02-28,5.33,5.25,5.5,96,5.31,1620.0
2024-02-29,5.33,5.25,5.5,96,5.32,1951.0
2024-03-01,5.33,5.25,5.5,84,5.31,1783.0
2024-03-04,5.33,5.25,5.5,91,5.31,1858.0
2024-03-05,5.33,5.25,5.5,101,5.31,1807.0
2024-03-06,5.33,5.25,5.5,99,5.31,1707.0
2024-03-07,5.33,5.25,5.5,98,5.31,1793.0
2024-03-08,5.33,5.25,5.5,85,5.31,1782.0
2024-03-11,5.33,5.25,5.5,85,5.31,1804.0
2024-03-12,5.33,5.25,5.5,86,5.31,1699.0
2024-03-13,5.33,5.25,5.5,89,5.31,1784.0
2024-03-14,5.33,5.25,5.5,95,5.31,1782.0
2024-03-15,5.33,5.25,5.5,90,5.31,1803.0
2024-03-18,5.33,5.25,5.5,91,5.31,1788.0
2024-03-19,5.33,5.25,5.5,89,5.31,1813.0
2024-03-20,5.33,5.25,5.5,91,5.31,1736.0
2024-03-21,5.33,5.25,5.5,93,5.31,1849.0
2024-03-22,5.33,5.25,5.5,91,5.31,1796.0
2024-03-25,5.33,5.25,5.5,85,5.31,1820.0
2024-03-26,5.33,5.25,5.5,85,5.32,1849.0
2024-03-27,5.33,5.25,5.5,82,5.33,1807.0
2024-03-28,5.33,5.25,5.5,53,5.34,1911.0
2024-03-29,5.33,5.25,5.5,62,,
2024-04-01,5.33,5.25,5.5,93,5.35,2023.0
2024-04-02,5.33,5.25,5.5,94,5.34,1927.0
2024-04-03,5.33,5.25,5.5,93,5.32,1985.0
2024-04-04,5.33,5.25,5.5,93,5.32,1974.0
2024-04-05,5.33,5.25,5.5,88,5.32,1898.0
2024-04-08,5.33,5.25,5.5,82,5.31,1807.0
2024-04-09,5.33,5.25,5.5,85,5.31,1826.0
2024-04-10,5.33,5.25,5.5,86,5.31,1909.0
2024-04-11,5.33,5.25,5.5,92,5.31,1857.0
2024-04-12,5.33,5.25,5.5,88,5.31,1805.0
2024-04-15,5.33,5.25,5.5,83,5.32,1938.0
2024-04-16,5.33,5.25,5.5,83,5.31,1809.0
2024-04-17,5.33,5.25,5.5,80,5.31,1731.0
2024-04-18,5.33,5.25,5.5,82,5.3,1764.0
2024-04-19,5.33,5.25,5.5,85,5.31,1780.0
2024-04-22,5.33,5.25,5.5,79,5.31,1746.0
2024-04-23,5.33,5.25,5.5,74,5.31,1757.0
2024-04-24,5.33,5.25,5.5,71,5.31,1778.0
2024-04-25,5.33,5.25,5.5,89,5.31,1774.0
2024-04-26,5.33,5.25,5.5,94,5.32,1795.0
2024-04-29,5.33,5.25,5.5,78,5.32,1781.0
2024-04-30,5.33,5.25,5.5,82,5.34,1831.0
2024-05-01,5.33,5.25,5.5,83,5.32,1955.0
2024-05-02,5.33,5.25,5.5,83,5.31,1838.0
2024-05-03,5.33,5.25,5.5,82,5.31,1863.0
2024-05-06,5.33,5.25,5.5,75,5.31,1855.0
2024-05-07,5.33,5.25,5.5,78,5.31,1864.0
2024-05-08,5.33,5.25,5.5,77,5.31,1845.0
2024-05-09,5.33,5.25,5.5,79,5.31,1871.0
2024-05-10,5.33,5.25,5.5,79,5.31,1816.0
2024-05-13,5.33,5.25,5.5,76,5.31,1805.0
2024-05-14,5.33,5.25,5.5,76,5.31,1838.0
2024-05-15,5.33,5.25,5.5,78,5.31,1936.0
2024-05-16,5.33,5.25,5.5,78,5.31,2001.0
2024-05-17,5.33,5.25,5.5,77,5.31,1964.0
2024-05-20,5.33,5.25,5.5,75,5.31,1866.0
2024-05-21,5.33,5.25,5.5,76,5.31,1929.0
2024-05-22,5.33,5.25,5.5,80,5.31,1900.0
2024-05-23,5.33,5.25,5.5,82,5.31,1916.0
2024-05-24,5.33,5.25,5.5,90,5.32,1909.0
2024-05-28,5.33,5.25,5.5,88,5.32,2013.0
2024-05-29,5.33,5.25,5.5,88,5.33,1807.0
2024-05-30,5.33,5.25,5.5,96,5.33,1824.0
2024-05-31,5.33,5.25,5.5,99,5.34,2061.0
2024-06-03,5.33,5.25,5.5,93,5.35,1912.0
2024-06-04,5.33,5.25,5.5,100,5.33,2092.0
2024-06-05,5.33,5.25,5.5,105,5.33,1989.0
2024-06-06,5.33,5.25,5.5,105,5.33,2008.0
2024-06-07,5.33,5.25,5.5,100,5.33,1934.0
2024-06-10,5.33,5.25,5.5,93,5.32,1847.0
2024-06-11,5.33,5.25,5.5,95,5.32,1946.0
2024-06-12,5.33,5.25,5.5,93,5.31,1957.0
2024-06-13,5.33,5.25,5.5,99,5.31,2035.0
2024-06-14,5.33,5.25,5.5,87,5.31,1855.0
2024-06-17,5.33,5.25,5.5,83,5.33,1975.0
2024-06-18,5.33,5.25,5.5,85,5.33,2021.0
2024-06-20,5.33,5.25,5.5,83,5.32,2052.0
2024-06-21,5.33,5.25,5.5,78,5.31,1991.0
2024-06-24,5.33,5.25,5.5,76,5.31,1992.0
2024-06-25,5.33,5.25,5.5,85,5.33,1968.0
2024-06-26,5.33,5.25,5.5,82,5.34,1897.0
2024-06-27,5.33,5.25,5.5,83,5.34,1888.0
2024-06-28,5.33,5.25,5.5,67,5.33,2024.0
2024-07-01,5.33,5.25,5.5,85,5.4,1967.0
2024-07-02,5.33,5.25,5.5,85,5.35,2110.0
2024-07-03,5.33,5.25,5.5,88,5.33,2007.0
2024-07-05,5.33,5.25,5.5,89,5.32,2126.0
2024-07-08,5.33,5.25,5.5,80,5.32,2090.0
2024-07-09,5.33,5.25,5.5,84,5.34,1913.0
2024-07-10,5.33,5.25,5.5,88,5.34,1949.0
2024-07-11,5.33,5.25,5.5,90,5.34,1920.0
2024-07-12,5.33,5.25,5.5,88,5.34,1900.0
2024-07-15,5.33,5.25,5.5,88,5.34,2175.0
2024-07-16,5.33,5.25,5.5,85,5.35,2061.0
2024-07-17,5.33,5.25,5.5,80,5.35,2025.0
2024-07-18,5.33,5.25,5.5,83,5.34,2014.0
2024-07-19,5.33,5.25,5.5,79,5.34,1992.0
2024-07-22,5.33,5.25,5.5,84,5.33,2120.0
2024-07-23,5.33,5.25,5.5,85,5.34,2011.0
2024-07-24,5.33,5.25,5.5,86,5.34,2083.0
2024-07-25,5.33,5.25,5.5,88,5.35,2129.0
2024-07-26,5.33,5.25,5.5,85,5.35,2027.0
2024-07-29,5.33,5.25,5.5,90,5.33,1982.0
2024-07-30,5.33,5.25,5.5,87,5.33,1978.0
2024-07-31,5.33,5.25,5.5,78,5.38,2185.0
2024-08-01,5.33,5.25,5.5,83,5.35,2211.0
2024-08-02,5.33,5.25,5.5,89,5.35,2077.0
2024-08-05,5.33,5.25,5.5,86,5.32,2199.0
2024-08-06,5.33,5.25,5.5,89,5.33,2086.0
2024-08-07,5.33,5.25,5.5,96,5.33,2042.0
2024-08-08,5.33,5.25,5.5,106,5.34,2196.0
2024-08-09,5.33,5.25,5.5,104,5.34,2170.0
2024-08-12,5.33,5.25,5.5,102,5.34,2138.0
2024-08-13,5.33,5.25,5.5,100,5.34,2082.0
2024-08-14,5.33,5.25,5.5,99,5.33,2108.0
2024-08-15,5.33,5.25,5.5,96,5.35,2118.0
2024-08-16,5.33,5.25,5.5,93,5.32,2131.0
2024-08-19,5.33,5.25,5.5,95,5.32,2118.0
2024-08-20,5.33,5.25,5.5,90,5.32,2101.0
2024-08-21,5.33,5.25,5.5,93,5.31,1975.0
2024-08-22,5.33,5.25,5.5,91,5.31,2037.0
2024-08-23,5.33,5.25,5.5,93,5.33,2152.0
2024-08-26,5.33,5.25,5.5,94,5.34,2039.0
2024-08-27,5.33,5.25,5.5,86,5.35,2227.0
2024-08-28,5.33,5.25,5.5,95,5.35,2030.0
2024-08-29,5.33,5.25,5.5,97,5.33,2088.0
2024-08-30,5.33,5.25,5.5,89,5.32,2011.0
2024-09-03,5.33,5.25,5.5,100,5.34,2403.0
2024-09-04,5.33,5.25,5.5,103,5.35,2169.0
2024-09-05,5.33,5.25,5.5,102,5.35,2205.0
2024-09-06,5.33,5.25,5.5,100,5.34,2119.0
2024-09-09,5.33,5.25,5.5,95,5.34,2094.0
2024-09-10,5.33,5.25,5.5,101,5.33,2161.0
2024-09-11,5.33,5.25,5.5,105,5.32,2308.0
2024-09-12,5.33,5.25,5.5,111,5.33,2087.0
2024-09-13,5.33,5.25,5.5,105,5.33,2232.0
2024-09-16,5.33,5.25,5.5,96,5.38,2212.0
2024-09-17,5.33,5.25,5.5,103,5.38,2161.0
2024-09-18,5.33,5.25,5.5,102,5.33,2194.0
2024-09-19,4.83,4.75,5.0,90,4.82,2378.0
2024-09-20,4.83,4.75,5.0,93,4.83,2147.0
2024-09-23,4.83,4.75,5.0,95,4.83,2198.0
2024-09-24,4.83,4.75,5.0,94,4.84,2171.0
2024-09-25,4.83,4.75,5.0,103,4.84,2133.0
2024-09-26,4.83,4.75,5.0,100,4.83,2329.0
2024-09-27,4.83,4.75,5.0,94,4.84,2077.0
2024-09-30,4.83,4.75,5.0,73,4.96,2523.0
2024-10-01,4.83,4.75,5.0,85,5.05,2395.0
2024-10-02,4.83,4.75,5.0,81,4.92,2426.0
2024-10-03,4.83,4.75,5.0,81,4.85,2182.0
2024-10-04,4.83,4.75,5.0,84,4.83,2377.0
2024-10-07,4.83,4.75,5.0,86,4.83,2350.0
2024-10-08,4.83,4.75,5.0,90,4.84,2166.0
2024-10-09,4.83,4.75,5.0,94,4.83,2220.0
2024-10-10,4.83,4.75,5.0,98,4.82,2198.0
2024-10-11,4.83,4.75,5.0,94,4.81,2058.0
2024-10-15,4.83,4.75,5.0,91,4.86,2272.0
2024-10-16,4.83,4.75,5.0,97,4.86,2317.0
2024-10-17,4.83,4.75,5.0,96,4.85,2334.0
2024-10-18,4.83,4.75,5.0,98,4.84,2168.0
2024-10-21,4.83,4.75,5.0,96,4.82,2322.0
2024-10-22,4.83,4.75,5.0,98,4.83,2226.0
2024-10-23,4.83,4.75,5.0,99,4.83,2129.0
2024-10-24,4.83,4.75,5.0,98,4.83,2246.0
2024-10-25,4.83,4.75,5.0,104,4.83,2188.0
2024-10-28,4.83,4.75,5.0,99,4.82,2194.0
2024-10-29,4.83,4.75,5.0,100,4.82,2116.0
2024-10-30,4.83,4.75,5.0,104,4.81,2045.0
2024-10-31,4.83,4.75,5.0,96,4.9,2244.0
2024-11-01,4.83,4.75,5.0,99,4.86,2164.0
2024-11-04,4.83,4.75,5.0,98,4.82,2308.0
2024-11-05,4.83,4.75,5.0,104,4.82,2235.0
2024-11-06,4.83,4.75,5.0,102,4.81,2128.0
2024-11-07,4.83,4.75,5.0,100,4.82,2266.0
2024-11-08,4.58,4.5,4.75,103,4.6,2184.0
2024-11-12,4.58,4.5,4.75,101,4.6,2298.0
2024-11-13,4.58,4.5,4.75,105,4.59,2233.0
2024-11-14,4.58,4.5,4.75,107,4.58,2177.0
2024-11-15,4.58,4.5,4.75,104,4.57,2425.0
2024-11-18,4.58,4.5,4.75,103,4.57,2297.0
2024-11-19,4.58,4.5,4.75,99,4.57,2275.0
2024-11-20,4.58,4.5,4.75,102,4.56,2155.0
2024-11-21,4.58,4.5,4.75,101,4.57,2291.0
2024-11-22,4.58,4.5,4.75,100,4.57,2188.0
2024-11-25,4.58,4.5,4.75,103,4.58,2206.0
2024-11-26,4.58,4.5,4.75,101,4.58,2214.0
2024-11-27,4.58,4.5,4.75,103,4.57,2136.0
2024-11-29,4.58,4.5,4.75,95,4.59,2209.0
2024-12-02,4.58,4.5,4.75,92,4.64,2440.0
2024-12-03,4.58,4.5,4.75,94,4.64,2329.0
2024-12-04,4.58,4.5,4.75,99,4.59,2265.0
2024-12-05,4.58,4.5,4.75,109,4.59,2325.0
2024-12-06,4.58,4.5,4.75,107,4.6,2277.0
2024-12-09,4.58,4.5,4.75,100,4.63,2305.0
2024-12-10,4.58,4.5,4.75,102,4.64,2308.0
2024-12-11,4.58,4.5,4.75,106,4.62,2287.0
2024-12-12,4.58,4.5,4.75,111,4.62,2262.0
2024-12-13,4.58,4.5,4.75,102,4.6,2255.0
2024-12-16,4.58,4.5,4.75,99,4.65,2360.0
2024-12-17,4.58,4.5,4.75,103,4.62,2341.0
2024-12-18,4.58,4.5,4.75,112,4.57,2291.0
2024-12-19,4.33,4.25,4.5,104,4.3,2390.0
2024-12-20,4.33,4.25,4.5,113,4.3,2330.0
2024-12-23,4.33,4.25,4.5,119,4.31,2337.0
2024-12-24,4.33,4.25,4.5,118,4.4,2282.0
2024-12-26,4.33,4.25,4.5,128,4.53,2320.0
2024-12-27,4.33,4.25,4.5,119,4.46,2275.0
2024-12-30,4.33,4.25,4.5,108,4.37,2290.0
2024-12-31,4.33,4.25,4.5,53,4.49,2474.0
2025-01-02,4.33,4.25,4.5,106,4.4,2501.0
2025-01-03,4.33,4.25,4.5,102,4.31,2398.0
2025-01-06,4.33,4.25,4.5,98,4.27,2349.0
2025-01-07,4.33,4.25,4.5,99,4.27,2322.0
2025-01-08,4.33,4.25,4.5,105,4.29,2213.0
2025-01-09,4.33,4.25,4.5,104,4.3,2211.0
2025-01-10,4.33,4.25,4.5,102,4.3,2204.0
2025-01-13,4.33,4.25,4.5,104,4.29,2211.0
2025-01-14,4.33,4.25,4.5,112,4.28,2179.0
2025-01-15,4.33,4.25,4.5,108,4.28,2280.0
2025-01-16,4.33,4.25,4.5,105,4.29,2318.0
2025-01-17,4.33,4.25,4.5,102,4.29,2326.0
2025-01-21,4.33,4.25,4.5,100,4.29,2361.0
2025-01-22,4.33,4.25,4.5,101,4.3,2269.0
2025-01-23,4.33,4.25,4.5,93,4.35,2269.0
2025-01-24,4.33,4.25,4.5,97,4.34,2289.0
2025-01-27,4.33,4.25,4.5,97,4.34,2338.0
2025-01-28,4.33,4.25,4.5,95,4.35,2283.0
2025-01-29,4.33,4.25,4.5,92,4.35,2323.0
2025-01-30,4.33,4.25,4.5,103,4.36,2369.0
2025-01-31,4.33,4.25,4.5,83,4.38,2533.0
2025-02-03,4.33,4.25,4.5,94,4.35,2413.0
2025-02-04,4.33,4.25,4.5,104,4.33,2344.0
2025-02-05,4.33,4.25,4.5,105,4.33,2343.0
2025-02-06,4.33,4.25,4.5,105,4.36,2365.0
2025-02-07,4.33,4.25,4.5,107,4.35,2336.0
2025-02-10,4.33,4.25,4.5,96,4.35,2316.0
2025-02-11,4.33,4.25,4.5,92,4.34,2321.0
2025-02-12,4.33,4.25,4.5,94,4.32,2313.0
2025-02-13,4.33,4.25,4.5,94,4.33,2329.0
2025-02-14,4.33,4.25,4.5,91,4.33,2319.0
2025-02-18,4.33,4.25,4.5,89,4.37,2452.0
2025-02-19,4.33,4.25,4.5,95,4.35,2394.0
2025-02-20,4.33,4.25,4.5,98,4.33,2403.0
2025-02-21,4.33,4.25,4.5,103,4.34,2400.0
2025-02-24,4.33,4.25,4.5,99,4.34,2425.0
2025-02-25,4.33,4.25,4.5,108,4.33,2467.0
2025-02-26,4.33,4.25,4.5,108,4.33,2487.0
2025-02-27,4.33,4.25,4.5,110,4.36,2557.0
2025-02-28,4.33,4.25,4.5,95,4.39,2684.0
2025-03-03,4.33,4.25,4.5,110,4.33,2636.0
2025-03-04,4.33,4.25,4.5,104,4.33,2562.0
2025-03-05,4.33,4.25,4.5,112,4.34,2525.0
2025-03-06,4.33,4.25,4.5,115,4.35,2558.0
2025-03-07,4.33,4.25,4.5,110,4.34,2477.0
2025-03-10,4.33,4.25,4.5,105,4.33,2505.0
2025-03-11,4.33,4.25,4.5,97,4.32,2502.0
2025-03-12,4.33,4.25,4.5,98,4.31,2489.0
2025-03-13,4.33,4.25,4.5,105,4.3,2462.0
2025-03-14,4.33,4.25,4.5,104,4.3,2441.0
2025-03-17,4.33,4.25,4.5,106,4.32,2473.0
2025-03-18,4.33,4.25,4.5,113,4.31,2465.0
2025-03-19,4.33,4.25,4.5,106,4.29,2376.0
2025-03-20,4.33,4.25,4.5,109,4.29,2438.0
2025-03-21,4.33,4.25,4.5,106,4.3,2444.0
2025-03-24,4.33,4.25,4.5,107,4.31,2457.0
2025-03-25,4.33,4.25,4.5,105,4.33,2460.0
2025-03-26,4.33,4.25,4.5,104,4.35,2472.0
2025-03-27,4.33,4.25,4.5,109,4.36,2468.0
2025-03-28,4.33,4.25,4.5,112,4.34,2440.0
2025-03-31,4.33,4.25,4.5,78,4.41,2636.0
2025-04-01,4.33,4.25,4.5,112,4.39,2589.0
2025-04-02,4.33,4.25,4.5,104,4.37,2555.0
2025-04-03,4.33,4.25,4.5,103,4.39,2577.0
2025-04-04,4.33,4.25,4.5,95,4.35,2587.0
2025-04-07,4.33,4.25,4.5,93,4.33,2630.0
2025-04-08,4.33,4.25,4.5,93,4.4,2658.0
2025-04-09,4.33,4.25,4.5,92,4.42,2663.0
2025-04-10,4.33,4.25,4.5,98,4.37,2657.0
2025-04-11,4.33,4.25,4.5,105,4.33,2584.0
2025-04-14,4.33,4.25,4.5,99,4.33,2561.0
2025-04-15,4.33,4.25,4.5,96,4.36,2526.0
2025-04-16,4.33,4.25,4.5,88,4.31,2513.0
2025-04-17,4.33,4.25,4.5,99,4.32,2505.0
2025-04-18,4.33,4.25,4.5,94,,
2025-04-21,4.33,4.25,4.5,96,4.32,2469.0
2025-04-22,4.33,4.25,4.5,99,4.3,2527.0
2025-04-23,4.33,4.25,4.5,107,4.28,2475.0
2025-04-24,4.33,4.25,4.5,103,4.29,2542.0
2025-04-25,4.33,4.25,4.5,112,4.33,2565.0
2025-04-28,4.33,4.25,4.5,105,4.36,2567.0
2025-04-29,4.33,4.25,4.5,104,4.36,2623.0
2025-04-30,4.33,4.25,4.5,92,4.41,2830.0
2025-05-01,4.33,4.25,4.5,107,4.39,2789.0
2025-05-02,4.33,4.25,4.5,111,4.36,2694.0
2025-05-05,4.33,4.25,4.5,117,4.33,2632.0
2025-05-06,4.33,4.25,4.5,120,4.32,2620.0
2025-05-07,4.33,4.25,4.5,111,4.3,2535.0
2025-05-08,4.33,4.25,4.5,113,4.29,2545.0
2025-05-09,4.33,4.25,4.5,107,4.28,2479.0
2025-05-12,4.33,4.25,4.5,105,4.28,2535.0
2025-05-13,4.33,4.25,4.5,113,4.3,2565.0
2025-05-14,4.33,4.25,4.5,112,4.29,2543.0
2025-05-15,4.33,4.25,4.5,120,4.31,2686.0
2025-05-16,4.33,4.25,4.5,117,4.3,2617.0
2025-05-19,4.33,4.25,4.5,116,4.29,2570.0
2025-05-20,4.33,4.25,4.5,111,4.27,2554.0
2025-05-21,4.33,4.25,4.5,118,4.26,2517.0
2025-05-22,4.33,4.25,4.5,118,4.26,2540.0
2025-05-23,4.33,4.25,4.5,123,4.26,2542.0
2025-05-27,4.33,4.25,4.5,115,4.31,2655.0
2025-05-28,4.33,4.25,4.5,108,4.33,2605.0
2025-05-29,4.33,4.25,4.5,114,4.33,2652.0
2025-05-30,4.33,4.25,4.5,105,4.35,2641.0
2025-06-02,4.33,4.25,4.5,113,4.35,2771.0
2025-06-03,4.33,4.25,4.5,120,4.32,2711.0
2025-06-04,4.33,4.25,4.5,114,4.28,2625.0
2025-06-05,4.33,4.25,4.5,118,4.29,2685.0
2025-06-06,4.33,4.25,4.5,118,4.29,2667.0
2025-06-09,4.33,4.25,4.5,114,4.29,2643.0
2025-06-10,4.33,4.25,4.5,112,4.28,2634.0
2025-06-11,4.33,4.25,4.5,111,4.28,2600.0
2025-06-12,4.33,4.25,4.5,114,4.28,2663.0
2025-06-13,4.33,4.25,4.5,110,4.28,2644.0
2025-06-16,4.33,4.25,4.5,101,4.32,2697.0
2025-06-17,4.33,4.25,4.5,104,4.31,2692.0
2025-06-18,4.33,4.25,4.5,109,4.28,2639.0
2025-06-20,4.33,4.25,4.5,118,4.29,2700.0
2025-06-23,4.33,4.25,4.5,115,4.29,2740.0
2025-06-24,4.33,4.25,4.5,115,4.3,2733.0
2025-06-25,4.33,4.25,4.5,128,4.36,2772.0
2025-06-26,4.33,4.25,4.5,124,4.4,2802.0
2025-06-27,4.33,4.25,4.5,122,4.39,2775.0
2025-06-30,4.33,4.25,4.5,79,4.45,2946.0
2025-07-01,4.33,4.25,4.5,133,4.44,2920.0
2025-07-02,4.33,4.25,4.5,129,4.4,2882.0
2025-07-03,4.33,4.25,4.5,129,4.35,2846.0
2025-07-07,4.33,4.25,4.5,118,4.33,2814.0
2025-07-08,4.33,4.25,4.5,117,4.34,2765.0
2025-07-09,4.33,4.25,4.5,105,4.32,2752.0
2025-07-10,4.33,4.25,4.5,106,4.31,2735.0
2025-07-11,4.33,4.25,4.5,103,4.31,2754.0
2025-07-14,4.33,4.25,4.5,104,4.33,2759.0
2025-07-15,4.33,4.25,4.5,101,4.37,2784.0
2025-07-16,4.33,4.25,4.5,115,4.34,2771.0
2025-07-17,4.33,4.25,4.5,113,4.34,2743.0
2025-07-18,4.33,4.25,4.5,110,4.3,2775.0
2025-07-21,4.33,4.25,4.5,108,4.28,2688.0
2025-07-22,4.33,4.25,4.5,107,4.28,2692.0
2025-07-23,4.33,4.25,4.5,115,4.28,2659.0
2025-07-24,4.33,4.25,4.5,109,4.3,2725.0
2025-07-25,4.33,4.25,4.5,114,4.36,2766.0
2025-07-28,4.33,4.25,4.5,112,4.36,2783.0
2025-07-29,4.33,4.25,4.5,112,4.36,2754.0
2025-07-30,4.33,4.25,4.5,105,4.32,2727.0
2025-07-31,4.33,4.25,4.5,92,4.39,2933.0
2025-08-01,4.33,4.25,4.5,129,4.34,2885.0
2025-08-04,4.33,4.25,4.5,117,4.33,2828.0
2025-08-05,4.33,4.25,4.5,115,4.34,2863.0
2025-08-06,4.33,4.25,4.5,112,4.34,2866.0
2025-08-07,4.33,4.25,4.5,108,4.35,2855.0
2025-08-08,4.33,4.25,4.5,115,4.35,2817.0
2025-08-11,4.33,4.25,4.5,115,4.34,2796.0
2025-08-12,4.33,4.25,4.5,114,4.36,2804.0
2025-08-13,4.33,4.25,4.5,116,4.33,2796.0
2025-08-14,4.33,4.25,4.5,114,4.34,2767.0
2025-08-15,4.33,4.25,4.5,112,4.36,2821.0
2025-08-18,4.33,4.25,4.5,112,4.34,2810.0
2025-08-19,4.33,4.25,4.5,109,4.33,2760.0
2025-08-20,4.33,4.25,4.5,115,4.31,2704.0
2025-08-21,4.33,4.25,4.5,117,4.32,2702.0
2025-08-22,4.33,4.25,4.5,114,4.36,2738.0
2025-08-25,4.33,4.25,4.5,113,4.37,2825.0
2025-08-26,4.33,4.25,4.5,113,4.38,2820.0
2025-08-27,4.33,4.25,4.5,114,4.36,2889.0
2025-08-28,4.33,4.25,4.5,115,4.34,2870.0
2025-08-29,4.33,4.25,4.5,115,4.34,2880.0
2025-09-02,4.33,4.25,4.5,120,4.39,2947.0
2025-09-03,4.33,4.25,4.5,119,4.39,2880.0
2025-09-04,4.33,4.25,4.5,118,4.41,2834.0
2025-09-05,4.33,4.25,4.5,116,4.42,2847.0
2025-09-08,4.33,4.25,4.5,110,4.4,2872.0
2025-09-09,4.33,4.25,4.5,112,4.4,2863.0
2025-09-10,4.33,4.25,4.5,111,4.39,2800.0
2025-09-11,4.33,4.25,4.5,117,4.41,2828.0
2025-09-12,4.33,4.25,4.5,106,4.42,2830.0
2025-09-15,4.33,4.25,4.5,91,4.51,2877.0
2025-09-16,4.33,4.25,4.5,98,4.39,2857.0
2025-09-17,4.33,4.25,4.5,96,4.38,2853.0
2025-09-18,4.08,4.0,4.25,92,4.14,2894.0
2025-09-19,4.08,4.0,4.25,95,4.14,2881.0
2025-09-22,4.09,4.0,4.25,93,4.14,2906.0
2025-09-23,4.09,4.0,4.25,95,4.12,2877.0
2025-09-24,4.09,4.0,4.25,88,4.13,2852.0
2025-09-25,4.09,4.0,4.25,92,4.18,2965.0
2025-09-26,4.09,4.0,4.25,95,4.16,2917.0
2025-09-29,4.09,4.0,4.25,89,4.13,2893.0
2025-09-30,4.09,4.0,4.25,92,4.24,3148.0
2025-10-01,4.09,4.0,4.25,103,4.2,3075.0
2025-10-02,4.09,4.0,4.25,102,4.2,3022.0
2025-10-03,4.09,4.0,4.25,95,4.18,3018.0
2025-10-06,4.09,4.0,4.25,84,4.15,2981.0
2025-10-07,4.09,4.0,4.25,81,4.14,2946.0
2025-10-08,4.1,4.0,4.25,76,4.12,2924.0
2025-10-09,4.1,4.0,4.25,84,4.13,2923.0
2025-10-10,4.1,4.0,4.25,87,4.15,2895.0
2025-10-14,4.1,4.0,4.25,80,4.19,2932.0
2025-10-15,4.1,4.0,4.25,81,4.29,3059.0
2025-10-16,4.11,4.0,4.25,85,4.3,3045.0
2025-10-17,4.11,4.0,4.25,79,4.18,3022.0
2025-10-20,4.11,4.0,4.25,82,4.16,2945.0
2025-10-21,4.11,4.0,4.25,85,4.23,2957.0
2025-10-22,4.11,4.0,4.25,91,4.21,2956.0
2025-10-23,4.11,4.0,4.25,93,4.24,3002.0
2025-10-24,4.11,4.0,4.25,90,4.24,3000.0
2025-10-27,4.12,4.0,4.25,88,4.27,3019.0
2025-10-28,4.12,4.0,4.25,92,4.31,3067.0
2025-10-29,4.12,4.0,4.25,98,4.27,3061.0
2025-10-30,3.87,3.75,4.0,104,4.04,3098.0
2025-10-31,3.86,3.75,4.0,107,4.22,3211.0
2025-11-03,3.87,3.75,4.0,98,4.13,3237.0
2025-11-04,3.87,3.75,4.0,94,4.0,3147.0
2025-11-05,3.87,3.75,4.0,94,3.91,3145.0
2025-11-06,3.87,3.75,4.0,87,3.92,3156.0
2025-11-07,3.87,3.75,4.0,83,3.93,3069.0
2025-11-10,3.87,3.75,4.0,77,3.95,3135.0
2025-11-12,3.87,3.75,4.0,77,3.98,3209.0
2025-11-13,3.88,3.75,4.0,78,4.0,3197.0
2025-11-14,3.88,3.75,4.0,72,3.95,3195.0
2025-11-17,3.88,3.75,4.0,75,4.0,3278.0
2025-11-18,3.88,3.75,4.0,78,3.94,3220.0
2025-11-19,3.88,3.75,4.0,76,3.91,3179.0
2025-11-20,3.88,3.75,4.0,75,3.91,3170.0
2025-11-21,3.88,3.75,4.0,81,3.93,3203.0
2025-11-24,3.88,3.75,4.0,73,3.96,3232.0
2025-11-25,3.88,3.75,4.0,78,4.01,3317.0
2025-11-26,3.88,3.75,4.0,89,4.05,3297.0
2025-11-28,3.89,3.75,4.0,90,4.12,3361.0
2025-12-01,3.89,3.75,4.0,85,4.12,3454.0
2025-12-02,3.89,3.75,4.0,91,4.01,3407.0
2025-12-03,3.89,3.75,4.0,85,3.95,3360.0
2025-12-04,3.89,3.75,4.0,87,3.92,3300.0
2025-12-05,3.89,3.75,4.0,88,3.93,3221.0
2025-12-08,3.89,3.75,4.0,84,3.95,3209.0
2025-12-09,3.89,3.75,4.0,85,3.93,3244.0
2025-12-10,3.89,3.75,4.0,98,3.9,3222.0
2025-12-11,3.64,3.5,3.75,99,3.66,3299.0
2025-12-12,3.64,3.5,3.75,102,3.67,3261.0
2025-12-15,3.64,3.5,3.75,97,3.75,3270.0
2025-12-16,3.64,3.5,3.75,95,3.69,3319.0
2025-12-17,3.64,3.5,3.75,94,3.69,3289.0
2025-12-18,3.64,3.5,3.75,88,3.66,3273.0
2025-12-19,3.64,3.5,3.75,93,3.66,3238.0
2025-12-22,3.64,3.5,3.75,87,3.68,3253.0
2025-12-23,3.64,3.5,3.75,85,3.66,3255.0
2025-12-24,3.64,3.5,3.75,88,3.66,3194.0
2025-12-26,3.64,3.5,3.75,101,3.76,3227.0
2025-12-29,3.64,3.5,3.75,87,3.77,3346.0
2025-12-30,3.64,3.5,3.75,87,3.71,3321.0
2025-12-31,3.64,3.5,3.75,74,3.87,3485.0
2026-01-02,3.64,3.5,3.75,92,3.75,3508.0
2026-01-05,3.64,3.5,3.75,88,3.7,3441.0
2026-01-06,3.64,3.5,3.75,88,3.66,3390.0
2026-01-07,3.64,3.5,3.75,83,3.65,3267.0
2026-01-08,3.64,3.5,3.75,87,3.64,3249.0
2026-01-09,3.64,3.5,3.75,93,3.64,3166.0
2026-01-12,3.64,3.5,3.75,93,3.64,3128.0
2026-01-13,3.64,3.5,3.75,92,3.65,3177.0
2026-01-14,3.64,3.5,3.75,94,3.64,3148.0
2026-01-15,3.64,3.5,3.75,91,3.66,3201.0
2026-01-16,3.64,3.5,3.75,84,3.65,3163.0
2026-01-20,3.64,3.5,3.75,83,3.64,3162.0
2026-01-21,3.64,3.5,3.75,95,3.63,3087.0
2026-01-22,3.64,3.5,3.75,89,3.64,3121.0
2026-01-23,3.64,3.5,3.75,99,3.65,3137.0
2026-01-26,3.64,3.5,3.75,83,3.66,3143.0
2026-01-27,3.64,3.5,3.75,88,3.66,3145.0
2026-01-28,3.64,3.5,3.75,89,3.64,3146.0
2026-01-29,3.64,3.5,3.75,104,3.65,3166.0
2026-01-30,3.64,3.5,3.75,101,3.68,3275.0
2026-02-02,3.64,3.5,3.75,93,3.69,3307.0
2026-02-03,3.64,3.5,3.75,107,3.69,3268.0
2026-02-04,3.64,3.5,3.75,109,3.65,3310.0
2026-02-05,3.64,3.5,3.75,110,3.65,3228.0
2026-02-06,3.64,3.5,3.75,106,3.64,3195.0
2026-02-09,3.64,3.5,3.75,92,3.63,3132.0
2026-02-10,3.64,3.5,3.75,104,3.65,3194.0
2026-02-11,3.64,3.5,3.75,101,3.65,3167.0
2026-02-12,3.64,3.5,3.75,97,3.65,3205.0
2026-02-13,3.64,3.5,3.75,90,3.66,3169.0
2026-02-17,3.64,3.5,3.75,97,3.71,3254.0
2026-02-18,3.64,3.5,3.75,104,3.73,3258.0
2026-02-19,3.64,3.5,3.75,100,3.67,3238.0
2026-02-20,3.64,3.5,3.75,101,3.66,3224.0
2026-02-23,3.64,3.5,3.75,97,3.66,3197.0
2026-02-24,3.64,3.5,3.75,106,3.67,3239.0
2026-02-25,3.64,3.5,3.75,108,3.67,3232.0
2026-02-26,3.64,3.5,3.75,108,3.67,3262.0
2026-02-27,3.64,3.5,3.75,107,3.68,3252.0
2026-03-02,3.64,3.5,3.75,97,3.71,3404.0
2026-03-03,3.64,3.5,3.75,101,3.7,3310.0
2026-03-04,3.64,3.5,3.75,106,3.67,3292.0
2026-03-05,3.64,3.5,3.75,106,3.66,3294.0
2026-03-06,3.64,3.5,3.75,102,3.65,3198.0
2026-03-09,3.64,3.5,3.75,99,3.65,3173.0
2026-03-10,3.64,3.5,3.75,104,3.64,3200.0
2026-03-11,3.64,3.5,3.75,97,3.64,3175.0
2026-03-12,3.64,3.5,3.75,99,3.65,3112.0
2026-03-13,3.64,3.5,3.75,92,3.65,3164.0
2026-03-16,3.64,3.5,3.75,88,3.7,3178.0
2026-03-17,3.64,3.5,3.75,93,3.65,3146.0
2026-03-18,3.64,3.5,3.75,93,3.62,3092.0
2026-03-19,3.64,3.5,3.75,103,3.62,3121.0
2026-03-20,3.64,3.5,3.75,103,3.62,3102.0
2026-03-23,3.64,3.5,3.75,97,3.62,3013.0
2026-03-24,3.64,3.5,3.75,96,3.63,3014.0
2026-03-25,3.64,3.5,3.75,93,3.64,3063.0
2026-03-26,3.64,3.5,3.75,101,3.65,3113.0
2026-03-27,3.64,3.5,3.75,105,3.63,3037.0
2026-03-30,3.64,3.5,3.75,105,3.63,3082.0
2026-03-31,3.64,3.5,3.75,86,3.68,3280.0
2026-04-01,3.64,3.5,3.75,110,3.65,3265.0
2026-04-02,3.64,3.5,3.75,105,3.66,3251.0
2026-04-03,3.64,3.5,3.75,106,,
2026-04-06,3.64,3.5,3.75,103,3.65,3217.0
2026-04-07,3.64,3.5,3.75,106,3.62,3260.0
2026-04-08,3.64,3.5,3.75,106,3.59,3169.0
2026-04-09,3.64,3.5,3.75,105,3.57,3147.0
2026-04-10,3.64,3.5,3.75,106,3.61,3063.0
2026-04-13,3.64,3.5,3.75,97,3.63,3135.0
2026-04-14,3.64,3.5,3.75,94,3.66,3116.0
2026-04-15,3.64,3.5,3.75,83,3.72,3117.0
2026-04-16,3.64,3.5,3.75,85,3.67,3107.0
2026-04-17,3.64,3.5,3.75,85,3.65,3042.0
2026-04-20,3.64,3.5,3.75,69,3.63,3010.0
2026-04-21,3.64,3.5,3.75,85,3.63,3021.0
2026-04-22,3.64,3.5,3.75,89,3.64,2960.0
2026-04-23,3.64,3.5,3.75,93,3.65,3011.0
2026-04-24,3.64,3.5,3.75,97,3.66,3047.0
2026-04-27,3.64,3.5,3.75,85,3.66,3058.0
2026-04-28,3.64,3.5,3.75,82,3.64,3021.0
2026-04-29,3.64,3.5,3.75,87,3.63,2964.0
2026-04-30,3.64,3.5,3.75,90,3.66,3275.0
2026-05-01,3.64,3.5,3.75,115,3.64,3150.0
2026-05-04,3.64,3.5,3.75,111,3.63,3113.0
2026-05-05,3.64,3.5,3.75,113,3.62,3148.0
2026-05-06,3.64,3.5,3.75,118,3.61,3129.0
2026-05-07,3.63,3.5,3.75,119,3.6,3106.0
2026-05-08,3.63,3.5,3.75,123,3.6,3087.0
2026-05-11,3.63,3.5,3.75,114,3.6,3090.0
2026-05-12,3.63,3.5,3.75,118,3.6,3097.0
2026-05-13,3.63,3.5,3.75,123,3.59,3119.0
2026-05-14,3.63,3.5,3.75,120,3.56,3115.0
2026-05-15,3.63,3.5,3.75,123,3.55,3157.0
2026-05-18,3.63,3.5,3.75,118,3.53,3159.0
2026-05-19,3.62,3.5,3.75,119,3.51,3111.0
2026-05-20,3.62,3.5,3.75,119,3.5,3082.0
2026-05-21,3.62,3.5,3.75,120,3.51,3077.0
2026-05-22,3.62,3.5,3.75,121,3.55,3078.0
2026-05-26,3.62,3.5,3.75,113,3.63,3127.0
2026-05-27,3.62,3.5,3.75,106,3.63,3176.0
2026-05-28,3.62,3.5,3.75,106,3.62,3139.0
2026-05-29,3.62,3.5,3.75,109,3.63,3201.0
2026-06-01,3.62,3.5,3.75,123,3.65,3224.0
2026-06-02,3.62,3.5,3.75,133,3.63,3148.0
2026-06-03,3.62,3.5,3.75,119,3.61,3098.0
2026-06-04,3.62,3.5,3.75,121,3.62,3147.0
2026-06-05,3.62,3.5,3.75,117,3.63,3131.0
2026-06-08,3.62,3.5,3.75,108,3.63,3115.0
2026-06-09,3.62,3.5,3.75,106,3.6,3063.0
2026-06-10,3.62,3.5,3.75,106,3.59,3023.0
2026-06-11,3.62,3.5,3.75,106,3.6,3061.0
2026-06-12,3.62,3.5,3.75,110,3.65,3059.0
2026-06-15,3.63,3.5,3.75,102,3.69,3147.0
2026-06-16,3.63,3.5,3.75,106,3.63,3137.0
2026-06-17,3.63,3.5,3.75,107,3.63,3114.0
2026-06-18,3.63,3.5,3.75,115,3.62,3148.0
2026-06-22,3.63,3.5,3.75,115,3.61,3073.0
2026-06-23,3.63,3.5,3.75,109,3.62,3105.0
2026-06-24,3.63,3.5,3.75,113,3.62,3116.0
2026-06-25,3.63,3.5,3.75,120,3.64,3145.0
2026-06-26,3.63,3.5,3.75,120,3.62,3171.0
2026-06-29,3.63,3.5,3.75,123,3.62,3126.0
2026-06-30,3.63,3.5,3.75,98,3.68,3418.0
2026-07-01,3.63,3.5,3.75,119,3.66,3321.0
2026-07-02,3.63,3.5,3.75,121,3.64,3208.0
2026-07-03,3.63,3.5,3.75,121,,
2026-07-06,3.63,3.5,3.75,117,3.63,3212.0
2026-07-07,3.63,3.5,3.75,117,3.62,3154.0
2026-07-08,3.62,3.5,3.75,131,3.58,3158.0
2026-07-09,3.62,3.5,3.75,126,3.53,3126.0
2026-07-10,3.62,3.5,3.75,126,3.55,3056.0
2026-07-13,3.62,3.5,3.75,112,3.6,3096.0
2026-07-14,3.63,3.5,3.75,111,3.63,3092.0
2026-07-15,3.63,3.5,3.75,109,3.64,3104.0
2026-07-16,3.63,3.5,3.75,113,3.62,3038.0
2026-07-17,3.63,3.5,3.75,111,3.59,3030.0
2026-07-20,3.63,3.5,3.75,106,3.57,3012.0
2026-07-21,3.63,3.5,3.75,106,3.61,2975.0
2026-07-22,3.63,3.5,3.75,102,3.62,3026.0
2026-07-23,3.63,3.5,3.75,104,3.64,2971.0
2026-07-24,3.63,3.5,3.75,106,3.64,2979.0
2026-07-27,3.63,3.5,3.75,104,3.64,2953.0
2026-07-28,3.63,3.5,3.75,106,3.65,2977.0
2026-07-29,3.63,3.5,3.75,101,3.65,3032.0
2026-07-30,3.63,3.5,3.75,121,3.65,3011.0
2026-07-31,3.63,3.5,3.75,108,3.66,3205.0
2026-08-03,3.63,3.5,3.75,111,3.65,3055.0
2026-08-04,3.63,3.5,3.75,117,3.66,3036.0
2026-08-05,3.63,3.5,3.75,114,3.64,2989.0
2026-08-06,3.63,3.5,3.75,113,3.65,3055.0
2026-08-07,3.63,3.5,3.75,117,3.62,2977.0
2026-08-10,3.63,3.5,3.75,108,3.63,2964.0
2026-08-11,3.63,3.5,3.75,109,3.64,2961.0
2026-08-12,3.63,3.5,3.75,106,3.62,2943.0
2026-08-13,3.63,3.5,3.75,106,3.62,2932.0
2026-08-14,3.63,3.5,3.75,99,3.62,2957.0
2026-08-17,3.63,3.5,3.75,93,3.66,3068.0
2026-08-18,3.63,3.5,3.75,89,3.65,3010.0
2026-08-19,3.63,3.5,3.75,95,3.62,2923.0
2026-08-20,3.63,3.5,3.75,102,3.63,2922.0
2026-08-21,3.63,3.5,3.75,96,3.65,2952.0
2026-08-24,3.63,3.5,3.75,99,3.65,2919.0
2026-08-25,3.63,3.5,3.75,109,3.66,2916.0
2026-08-26,3.63,3.5,3.75,112,3.64,2859.0
2026-08-27,3.63,3.5,3.75,111,3.64,2836.0
2026-08-28,3.63,3.5,3.75,123,3.65,2808.0
2026-08-31,3.63,3.5,3.75,105,3.68,3056.0
2026-09-01,3.63,3.5,3.75,114,3.66,2912.0
2026-09-02,3.63,3.5,3.75,114,3.65,2882.0
2026-09-03,3.63,3.5,3.75,109,3.66,2949.0
2026-09-04,3.63,3.5,3.75,103,3.65,2888.0
2026-09-08,3.63,3.5,3.75,107,3.64,2904.0
2026-09-09,3.63,3.5,3.75,110,3.64,2859.0
2026-09-10,3.63,3.5,3.75,108,3.62,2921.0
2026-09-11,3.63,3.5,3.75,105,3.62,2867.0
2026-09-14,3.63,3.5,3.75,91,3.62,2861.0
2026-09-15,3.63,3.5,3.75,100,3.64,2952.0
2026-09-16,3.63,3.5,3.75,90,3.62,2931.0
2026-09-17,3.88,3.75,4.0,100,3.85,2992.0
2026-09-18,3.88,3.75,4.0,96,3.85,2955.0
2026-09-21,3.88,3.75,4.0,95,3.85,2912.0
2026-09-22,3.88,3.75,4.0,103,3.87,2940.0
2026-09-23,3.88,3.75,4.0,101,3.87,2946.0
2026-09-24,3.88,3.75,4.0,105,3.88,2990.0
""")
print('wrote data/nyfed_effr_sofr.csv')

In [ ]:
# Parameters (papermill overrides these)
data_csv = "data/nyfed_effr_sofr.csv"
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

rates = pd.read_csv(data_csv, parse_dates=["date"])
last = rates.iloc[-1]
out = {"meta": {
    "series": "SOFR, FX & Cross-Currency Curves",
    "episode": 1,
    "data_end": last.date.date().isoformat(),
    "quantlib_version": ql.__version__,
    "quotes_label": "real data: New York Fed (SOFR, EFFR); Federal Reserve Board (policy settings)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print(len(rates), "days,", rates.date.iloc[0].date(), "to", last.date.date(), "| QuantLib", ql.__version__)

## 2. The Fed's settings

The FOMC sets a **target range** for the federal funds rate, not a single rate. The Fed steers the market into the range with two rates it administers: **interest on reserve balances (IORB)**, paid to banks on their reserves at the Fed, and the **overnight reverse repo (ON RRP) offering rate**, available to a wider set of money-market lenders. Together they limit downward pressure on money-market rates and help keep the federal funds rate in the range (Board: *IORB FAQs*; *Overnight Reverse Repurchase Agreement Operations*).

The settings below are typed from the Board's *Implementation Note* of 16 September 2026, and the range is checked against the New York Fed data.

In [ ]:
IMPLEMENTATION_NOTE_2026_09_16 = {  # Federal Reserve Board, Implementation Note issued September 16, 2026
    "announced": "2026-09-16", "effective": "2026-09-17",
    "lower_pct": 3.75, "upper_pct": 4.00, "iorb_pct": 3.90, "onrrp_pct": 3.75,
}
p = IMPLEMENTATION_NOTE_2026_09_16
assert (last.target_low, last.target_high) == (p["lower_pct"], p["upper_pct"]), "range differs from the data"
out["policy"] = {**p,
    "width_bp": round((p["upper_pct"] - p["lower_pct"]) * 100),
    "iorb_above_lower_bp": round((p["iorb_pct"] - p["lower_pct"]) * 100),
    "onrrp_above_lower_bp": round((p["onrrp_pct"] - p["lower_pct"]) * 100),
    "rows": [
        {"setting": "Target range for the federal funds rate", "value": f"{p['lower_pct']:.2f}% to {p['upper_pct']:.2f}%"},
        {"setting": "Interest on reserve balances (IORB)", "value": f"{p['iorb_pct']:.2f}%"},
        {"setting": "Overnight reverse repo (ON RRP) offering rate", "value": f"{p['onrrp_pct']:.2f}%"},
        {"setting": "EFFR on the latest day in the data", "value": f"{last.effr:.2f}%"},
    ],
}
pd.DataFrame(out["policy"]["rows"])

## 3. Two overnight rates

| | EFFR | SOFR |
|---|---|---|
| Market | Overnight federal funds: unsecured loans between banks | Overnight repo: cash lent against US Treasury securities |
| Data | FR 2420 reports of overnight fed funds trades | Tri-party repo, GCF Repo and FICC-cleared bilateral (DVP) repo; the lowest 20% of DVP volume is removed |
| Method | Volume-weighted median, rounded to the basis point | Volume-weighted median, rounded to the basis point |
| Published | About 9:00am ET, next business day | About 8:00am ET, next business day |
| Calendar | Federal Reserve business days | US government securities market business days |

Source: New York Fed, *Additional Information about Reference Rates* and *SOFR*. In 2017 the Alternative Reference Rates Committee selected SOFR as the alternative to USD LIBOR (ARRC, *User's Guide to SOFR*, 2019); USD overnight index swaps now reference SOFR (CFTC, *Swaps Made Available to Trade*).

In [ ]:
both = rates.dropna(subset=["sofr"])
latest = both.iloc[-1]
out["latest"] = {
    "date": latest.date.date().isoformat(),
    "effr_pct": float(latest.effr), "sofr_pct": float(latest.sofr),
    "effr_volume_bn": float(latest.effr_volume_bn), "sofr_volume_bn": float(latest.sofr_volume_bn),
    "sofr_volume_trn": round(float(latest.sofr_volume_bn) / 1000, 1),
    "volume_ratio": round(float(latest.sofr_volume_bn / latest.effr_volume_bn)),
    "effr_above_lower_bp": round((latest.effr - latest.target_low) * 100),
}
# Days with an EFFR but no SOFR: the Fed was open, the government securities market was closed.
no_sofr = rates[rates.sofr.isna()]
out["calendars"] = {"days_effr_only": int(len(no_sofr)), "examples": [d.date().isoformat() for d in no_sofr.date[:3]]}
out["latest"], out["calendars"]

## 4. The target range since 2018

Every change of the range in the data, and the phases of the cycle.

In [ ]:
chg = rates[rates.target_low.diff().fillna(0) != 0][["date", "target_low", "target_high"]].reset_index(drop=True)
chg["move_bp"] = (rates.target_low.diff()[rates.target_low.diff().fillna(0) != 0].values * 100).round()
zero = chg[chg.target_low == 0].iloc[0]                           # March 2020: to 0-0.25%
hikes = chg[(chg.date > zero.date) & (chg.move_bp > 0) & (chg.date < "2024-01-01")]
peak = hikes.iloc[-1]
cuts = chg[(chg.date > peak.date) & (chg.move_bp < 0)]
latest_move = chg.iloc[-1]
prev_hike = chg[(chg.move_bp > 0) & (chg.date < latest_move.date)].iloc[-1]
out["history"] = {
    "start": rates.date.iloc[0].date().isoformat(),
    "start_lower_pct": float(rates.target_low.iloc[0]), "start_upper_pct": float(rates.target_high.iloc[0]),
    "n_changes": int(len(chg)),
    "zero_date": zero.date.date().isoformat(), "zero_upper_pct": float(zero.target_high),
    "hikes_n": int(len(hikes)), "hikes_first": hikes.date.iloc[0].date().isoformat(),
    "peak_date": peak.date.date().isoformat(), "peak_lower_pct": float(peak.target_low), "peak_upper_pct": float(peak.target_high),
    "hike_total_bp": float((peak.target_low - zero.target_low) * 100),
    "cuts_n": int(len(cuts)), "cuts_first": cuts.date.iloc[0].date().isoformat(), "cuts_last": cuts.date.iloc[-1].date().isoformat(),
    "cuts_low_lower_pct": float(cuts.target_low.iloc[-1]), "cuts_low_upper_pct": float(cuts.target_high.iloc[-1]),
    "latest_move_date": latest_move.date.date().isoformat(), "latest_move_bp": float(latest_move.move_bp),
    "previous_hike_date": prev_hike.date.date().isoformat(),
}
pd.Series(out["history"])

For the chart, one point per week (the last day of each week) for the range and EFFR. The x axis is in decimal years.

In [ ]:
decimal_year = lambda d: d.year + (d.dayofyear - 1) / (366 if d.is_leap_year else 365)
weekly = rates.set_index("date").resample("W-FRI").last().dropna(subset=["effr"]).reset_index()
out["chart"] = {
    "t": weekly.date.map(decimal_year).round(4).tolist(),
    "lower_pct": weekly.target_low.tolist(), "upper_pct": weekly.target_high.tolist(), "effr_pct": weekly.effr.tolist(),
    "ticks": [{"x": float(y), "label": str(y)} for y in range(2019, 2027)],
    "range": [round(decimal_year(rates.date.iloc[0]), 3), round(decimal_year(last.date), 3)],
}
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(weekly.date, weekly.target_low, weekly.target_high, step="post", alpha=.25, label="target range")
ax.plot(weekly.date, weekly.effr, lw=1.2, label="EFFR")
ax.set(title="FOMC target range and EFFR, % (Source: Federal Reserve Bank of New York)", ylabel="%")
ax.grid(alpha=.3); ax.legend()

## 5. EFFR inside the range

How many days has EFFR printed outside the target range?

In [ ]:
inside = rates.effr.between(rates.target_low, rates.target_high)
outside = rates[~inside]
o = outside.iloc[0]
out["inside"] = {
    "days": int(len(rates)), "days_inside": int(inside.sum()), "days_outside": int(len(outside)),
    "outside_date": o.date.date().isoformat(), "outside_effr_pct": float(o.effr),
    "outside_upper_pct": float(o.target_high), "outside_above_bp": round((o.effr - o.target_high) * 100),
}
print(f"EFFR inside the range on {inside.sum()} of {len(rates)} days")
outside

## 6. SOFR spikes

SOFR usually sits within a few basis points of EFFR, but it spikes. The largest spike came on 17 September 2019, when a corporate tax date and Treasury settlements drained reserves and cash from the repo market (Federal Reserve Board, FEDS Notes, *What Happened in Money Markets in September 2019?*, 2020). Smaller spikes cluster at the turn of the month, around Treasury settlement and reporting dates (same source; BIS *Quarterly Review*, September 2016).

A **month turn** here is the last or first day in the data of a calendar month.

In [ ]:
m = both.copy()
m["spread_bp"] = ((m.sofr - m.effr) * 100).round()
month = m.date.dt.to_period("M")
m["turn"] = (m.date == m.groupby(month).date.transform("max")) | (m.date == m.groupby(month).date.transform("min"))
threshold_bp = 20
big = m[m.spread_bp >= threshold_bp]
top = m.loc[m.spread_bp.idxmax()]
above = m[m.sofr > m.target_high]
recent = above[above.date >= "2025-01-01"]
out["spikes"] = {
    "threshold_bp": threshold_bp,
    "typical_abs_bp": float(m.spread_bp.abs().median()),
    "max_date": top.date.date().isoformat(), "max_sofr_pct": float(top.sofr), "max_effr_pct": float(top.effr),
    "max_spread_bp": float(top.spread_bp), "max_sofr_above_upper_bp": round((top.sofr - top.target_high) * 100),
    "n_big": int(len(big)), "n_big_turn": int(big.turn.sum()), "big_turn_share": float(big.turn.mean()),
    "turn_share_all_days": float(m.turn.mean()),
    "days_sofr_above_top": int(len(above)),
    "recent_first": recent.date.iloc[0].date().isoformat(), "recent_last": recent.date.iloc[-1].date().isoformat(),
    "recent_days_above_top": int(len(recent)),
}
out["spread_chart"] = {"t": m.date.map(decimal_year).round(4).tolist(), "spread_bp": m.spread_bp.tolist()}
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(m.date, m.spread_bp, lw=.8)
ax.set(title="SOFR minus EFFR, basis points, daily (Source: Federal Reserve Bank of New York)", ylabel="bp")
ax.grid(alpha=.3)
pd.Series({k: v for k, v in out["spikes"].items()})

## 7. A 360-day year

US money-market rates are quoted on a 360-day year (ARRC, *A User's Guide to SOFR*, 2019), Australian ones on 365. The same quoted rate therefore pays more interest per day in US dollars. One day's interest on USD 100 million at the latest SOFR:

In [ ]:
notional = 100_000_000
r = out["latest"]["sofr_pct"] / 100
start = ql.Date(1, 10, 2026)
one_day = {name: notional * r * dcc.yearFraction(start, start + 1)
           for name, dcc in {"act360": ql.Actual360(), "act365": ql.Actual365Fixed()}.items()}
out["daycount"] = {"notional": notional, "rate_pct": out["latest"]["sofr_pct"],
                   "act360": round(one_day["act360"], 2), "act365": round(one_day["act365"], 2),
                   "difference": round(one_day["act360"] - one_day["act365"], 2),
                   "extra_share": 365 / 360 - 1}
out["daycount"]

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")